# Transport Pathways Versus Dust Sources: a negative-control test of trajectory statistics, Riyadh

Google Colab notebook reproducing the paper. Run the cells in order; each step skips work already saved on Drive.
**Setup** → **1. Data** (ERA5, METAR, HYSPLIT; long, run once) → **2. Analysis code** → **3. Results** (tables and
figures in `07_Revision/`). Section 1 can be skipped if its outputs are already on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
PROJECT = "/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution"  # edit if needed
os.environ["RIYADH_PROJECT"] = PROJECT
PKG = "/content/rev_pkg"
os.makedirs(PKG, exist_ok=True)
sys.path.insert(0, PKG)

## 1. Data

**1.1 ERA5 download (asks for your CDS API key; resumes if interrupted)**

In [ ]:
import sys
import subprocess

print("INSTALLING / UPDATING ECMWF DATA STORES CLIENT")
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "ecmwf-datastores-client"], check=True)
from pathlib import Path
import calendar
import time
import random
import getpass
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
ERA5_DIR = PROJECT / "02_Meteorology" / "ERA5"
RAW_DIR = ERA5_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
START_YEAR = 2015
START_MONTH = 1
END_YEAR = 2025
END_MONTH = 12
AREA = [32.0, 35.0, 18.0, 55.0]
VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "10m_wind_gust_since_previous_post_processing",
    "2m_temperature",
    "2m_dewpoint_temperature",
    "surface_pressure",
    "boundary_layer_height",
    "volumetric_soil_water_layer_1",
    "total_precipitation",
    "surface_solar_radiation_downwards",
]
TIMES = [f"{hour:02d}:00" for hour in range(24)]
print("\n" + "=" * 70)
print("INITIALIZING ECMWF DATA STORES CLIENT")
from ecmwf.datastores import Client

print("\nEnter your CDS API key ONLY.")
print("Do NOT enter your email / User ID.")
API_KEY = getpass.getpass("\nCDS API key: ").strip()
if not API_KEY:
    raise RuntimeError("No API key supplied.")
client = Client(url="https://cds.climate.copernicus.eu/api", key=API_KEY)
print("\n✓ ECMWF client initialized.")
print("\n" + "=" * 70)
print("CHECKING ECMWF CLIENT")
print("\nclient.retrieve exists:", hasattr(client, "retrieve"))
if not hasattr(client, "retrieve"):
    raise RuntimeError("This ECMWF client does not provide client.retrieve().")
print("✓ Correct retrieve() method available.")
DATASET = "reanalysis-era5-single-levels"
print("\n" + "=" * 70)
print("ERA5 DOWNLOAD CONFIGURATION")
print(f"Dataset: {DATASET}")
print(f"Resume from: {START_YEAR}-{START_MONTH:02d}")
print(f"End: {END_YEAR}-{END_MONTH:02d}")
print("Already completed: 2015-01 through 2018-03")
print(f"Variables: {len(VARIABLES)}")
print(f"Hourly time steps: {len(TIMES)}")
print(f"Domain: N={AREA[0]}, W={AREA[1]}, S={AREA[2]}, E={AREA[3]}")
print("\nDIRECT GOOGLE DRIVE OUTPUT:")
print(RAW_DIR)
months = []
for year in range(START_YEAR, END_YEAR + 1):
    first_month = START_MONTH if year == START_YEAR else 1
    last_month = END_MONTH if year == END_YEAR else 12
    for month in range(first_month, last_month + 1):
        months.append((year, month))
print(f"\nMonths to process: {len(months)}")
successful = []
skipped = []
failed = []
start_time = time.time()
for counter, (year, month) in enumerate(months, start=1):
    month_str = f"{month:02d}"
    output_file = RAW_DIR / f"ERA5_{year}_{month_str}.nc"
    print("\n")
    print(f"MONTH {counter}/{len(months)}")
    print(f"ERA5 {year}-{month_str}")
    if output_file.exists():
        size_mb = output_file.stat().st_size / 1024**2
        if size_mb > 0.01:
            print("✓ EXISTING DRIVE FILE")
            print(f"Size: {size_mb:.2f} MB")
            skipped.append(f"{year}-{month_str}")
            continue
        else:
            print("⚠ Incomplete file detected.")
            print("Deleting it and redownloading.")
            output_file.unlink(missing_ok=True)
    num_days = calendar.monthrange(year, month)[1]
    days = [f"{day:02d}" for day in range(1, num_days + 1)]
    request = {
        "product_type": ["reanalysis"],
        "variable": VARIABLES,
        "year": [str(year)],
        "month": [month_str],
        "day": days,
        "time": TIMES,
        "area": AREA,
        "data_format": "netcdf",
        "download_format": "unarchived",
    }
    completed = False
    for attempt in range(1, 4):
        print(f"\nAttempt {attempt}/3")
        try:
            print("Submitting request...")
            result = client.retrieve(DATASET, request)
            print("✓ Request completed.")
            print("Saving directly to Google Drive...")
            if hasattr(result, "download"):
                result.download(str(output_file))
            else:
                result_path = Path(str(result))
                if result_path.exists():
                    if result_path.resolve() != output_file.resolve():
                        import shutil

                        shutil.copy2(result_path, output_file)
                else:
                    raise RuntimeError("ECMWF returned a result but no downloadable file was found.")
            if output_file.exists():
                size_mb = output_file.stat().st_size / 1024**2
                if size_mb > 0.01:
                    print("\n✓ DOWNLOAD SUCCESS")
                    print(f"File: {output_file}")
                    print(f"Size: {size_mb:.2f} MB")
                    successful.append(f"{year}-{month_str}")
                    completed = True
                    break
                else:
                    raise RuntimeError(f"File too small: {size_mb:.4f} MB")
            else:
                raise RuntimeError("File was not created on Google Drive.")
        except Exception as e:
            print("\n✗ DOWNLOAD ERROR:")
            print(str(e))
            output_file.unlink(missing_ok=True)
            if attempt < 3:
                wait = 20 * attempt + random.randint(5, 15)
                print(f"\nWaiting {wait} seconds before retry...")
                time.sleep(wait)
    if not completed:
        print(f"\n✗ FAILED: {year}-{month_str}")
        failed.append(f"{year}-{month_str}")
    time.sleep(3)
print("\n")
print("FINAL ERA5 DRIVE INVENTORY")
files = sorted(RAW_DIR.glob("ERA5_*.nc"))
total_bytes = 0
for f in files:
    size_mb = f.stat().st_size / 1024**2
    total_bytes += f.stat().st_size
    print(f"{f.name:25s}{size_mb:10.2f} MB")
total_gb = total_bytes / 1024**3
elapsed = time.time() - start_time
print("\n" + "=" * 70)
print("ERA5 DOWNLOAD SUMMARY")
print(f"Downloaded this run: {len(successful)}")
print(f"Skipped existing: {len(skipped)}")
print(f"Failed: {len(failed)}")
print(f"Total ERA5 files in Drive: {len(files)}")
print(f"Total storage: {total_gb:.3f} GB")
print(f"Runtime: {elapsed / 60:.1f} minutes")
if failed:
    print("\n" + "=" * 70)
    print("FAILED MONTHS")
    for item in failed:
        print(item)
    print("\nRun the SAME CELL again.")
    print("Existing completed months will automatically be skipped.")
else:
    print("\n" + "=" * 70)
    print("✓ ALL REQUESTED MONTHS COMPLETED")
print("\n" + "=" * 70)
print("FILES ARE HERE")
print(RAW_DIR)
print("\n✓ Directly on Google Drive")
print("✓ Resume-safe")
print("✓ Starting from April 2018")
print("✓ 2015–March 2018 are NOT touched")

**1.2 Unpack CDS archives into one NetCDF per month**

In [ ]:
import shutil
import tempfile
import zipfile
from pathlib import Path
import xarray as xr

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
ERA5_DIR = PROJECT / "02_Meteorology" / "ERA5"
RAW_DIR = ERA5_DIR / "raw"
FIXED_DIR = ERA5_DIR / "raw_merged"
FIXED_DIR.mkdir(parents=True, exist_ok=True)
DELETE_ORIGINALS = False
MAGIC = {b"PK\x03\x04": "zip", b"\x89HDF": "netcdf4", b"CDF\x01": "netcdf3", b"CDF\x02": "netcdf3"}

def sniff(path):
    head = open(path, "rb").read(4)
    for magic, kind in MAGIC.items():
        if head.startswith(magic):
            return kind
    return f"unknown ({head!r})"

def merge_zip(src, dst):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        with zipfile.ZipFile(src) as zf:
            zf.extractall(tmp)
        members = sorted(tmp.rglob("*.nc"))
        if not members:
            raise RuntimeError(f"No .nc members inside {src.name}")
        parts = [xr.open_dataset(m, engine="h5netcdf") for m in members]
        try:
            merged = xr.merge(parts, compat="override", join="exact")
        finally:
            for p in parts:
                p.close()
        encoding = {v: {"zlib": True, "complevel": 4} for v in merged.data_vars}
        merged.to_netcdf(dst, engine="h5netcdf", encoding=encoding)
        merged.close()
        return (len(members), list(merged.data_vars))

def main():
    files = sorted(RAW_DIR.glob("ERA5_*.nc"))
    print(f"Found {len(files)} files in {RAW_DIR}\n")
    kinds = {}
    for f in files:
        kinds.setdefault(sniff(f), []).append(f)
    print("File types detected:")
    for kind, group in kinds.items():
        print(f"  {kind:12s} {len(group)} files")
    if "zip" not in kinds:
        print("No ZIP-wrapped files found. This repair does not apply;")
        print("re-check the QC error message before continuing.")
        return
    done, failed, skipped = ([], [], [])
    for i, src in enumerate(kinds["zip"], start=1):
        dst = FIXED_DIR / src.name
        if dst.exists() and dst.stat().st_size > 1024:
            print(f"[{i}/{len(kinds['zip'])}] SKIP {src.name} (already merged)")
            skipped.append(src.name)
            continue
        try:
            n_members, dvars = merge_zip(src, dst)
            with xr.open_dataset(dst) as check:
                n_time = int(check.sizes.get("valid_time", check.sizes.get("time", 0)))
            print(
                f"[{i}/{len(kinds['zip'])}] OK   {src.name}  {n_members} streams -> {len(dvars)} vars, {n_time} steps"
            )
            done.append(src.name)
            if DELETE_ORIGINALS:
                src.unlink()
        except Exception as exc:
            print(f"[{i}/{len(kinds['zip'])}] FAIL {src.name}: {str(exc)[:160]}")
            dst.unlink(missing_ok=True)
            failed.append(src.name)
    print("\n" + "=" * 60)
    print(f"Merged   : {len(done)}")
    print(f"Skipped  : {len(skipped)}")
    print(f"Failed   : {len(failed)}")
    if failed:
        print("\nFailed files:")
        for name in failed:
            print(f"  {name}")
    print(f"\nMerged files -> {FIXED_DIR}")
    print("Point RAW_DIR in Notebook 02B at this folder, then re-run QC.")

if __name__ == "__main__":
    main()

**1.3 ERA5 daily fields**

In [ ]:
import shutil
import tempfile
import zipfile
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
ERA5_DIR = PROJECT / "02_Meteorology" / "ERA5"
RAW_DIR = ERA5_DIR / "raw_merged"
PROC_DIR = ERA5_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
RIYADH_LAT = 24.71
RIYADH_LON = 46.68
LOCAL_UTC_OFFSET_HOURS = 3
VAR_CANDIDATES = {
    "u10": ["u10", "10u", "u10m"],
    "v10": ["v10", "10v", "v10m"],
    "gust": ["fg10", "i10fg", "10fg", "p10fg"],
    "t2m": ["t2m", "2t"],
    "d2m": ["d2m", "2d"],
    "sp": ["sp", "surface_pressure"],
    "blh": ["blh", "pbl", "pblh"],
    "swvl1": ["swvl1", "volumetric_soil_water_layer_1"],
    "tp": ["tp", "total_precipitation"],
    "ssrd": ["ssrd", "surface_solar_radiation_downwards"],
}

def resolve_names(ds):
    resolved = {}
    missing = []
    for key, candidates in VAR_CANDIDATES.items():
        found = next((c for c in candidates if c in ds.variables), None)
        if found is None:
            missing.append(key)
        else:
            resolved[key] = found
    return (resolved, missing)

def get_time_dim(ds):
    for candidate in ["valid_time", "time"]:
        if candidate in ds.dims:
            return candidate
    raise KeyError(f"No time dimension found. Dims: {list(ds.dims)}")

def drop_expver(ds):
    if "expver" in ds.dims:
        ds = ds.reduce(np.nanmax, dim="expver", keep_attrs=True)
    elif "expver" in ds.coords:
        ds = ds.drop_vars("expver")
    return ds

def qc_inventory(raw_dir):
    print("\n" + "=" * 60)
    print("STAGE 1 - QC INVENTORY")
    expected_steps = {
        1: 744,
        2: 672,
        3: 744,
        4: 720,
        5: 744,
        6: 720,
        7: 744,
        8: 744,
        9: 720,
        10: 744,
        11: 720,
        12: 744,
    }

    def is_leap_year(year):
        return year % 4 == 0 and year % 100 != 0 or year % 400 == 0

    files = sorted(raw_dir.glob("ERA5_*.nc"))
    print(f"Found {len(files)} files in {raw_dir}\n")
    rows = []
    for f in files:
        year = int(f.name[5:9])
        month = int(f.name[10:12])
        if month == 2 and is_leap_year(year):
            expected = 696
        else:
            expected = expected_steps[month]
        size_mb = f.stat().st_size / 1024**2
        try:
            with xr.open_dataset(f) as ds:
                ds = drop_expver(ds)
                tdim = get_time_dim(ds)
                actual = int(ds.sizes[tdim])
                ok = actual == expected
                if ok:
                    status = "OK"
                    error = None
                else:
                    status = "TRUNCATED"
                    error = f"Expected {expected} steps, got {actual}"
        except Exception as e:
            ok = False
            status = "ERROR"
            error = str(e)
            actual = None
        rows.append(
            {
                "filename": f.name,
                "year": year,
                "month": month,
                "size_mb": size_mb,
                "expected_steps": expected,
                "actual_steps": actual,
                "status": status,
                "error": error,
                "ok": ok,
            }
        )
        print(f"{status: <8} {f.name:25s} {size_mb:8.1f} MB  steps={actual}/{expected}")
    qc_df = pd.DataFrame(rows)
    qc_df.to_csv(PROC_DIR / "ERA5_qc_report.csv", index=False)
    if qc_df["ok"].all():
        print("\n✓ All files passed QC.")
    else:
        print("\n✗ Some files failed QC. See report for details.")
    return qc_df

def saturation_vapour_pressure(temp_c):
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))

def add_derived(ds, names):
    u = ds[names["u10"]]
    v = ds[names["v10"]]
    ds["wind_speed"] = np.sqrt(u**2 + v**2)
    ds["wind_speed"].attrs = {"units": "m s-1", "long_name": "10 m wind speed"}
    ds["wind_dir"] = (270.0 - np.degrees(np.arctan2(v, u))) % 360.0
    ds["wind_dir"].attrs = {"units": "degrees", "long_name": "10 m wind direction (meteorological, from)"}
    t_c = ds[names["t2m"]] - 273.15
    td_c = ds[names["d2m"]] - 273.15
    ds["vpd"] = saturation_vapour_pressure(t_c) - saturation_vapour_pressure(td_c)
    ds["vpd"].attrs = {"units": "kPa", "long_name": "vapour pressure deficit"}
    return ds

def to_daily(ds, names):
    tdim = get_time_dim(ds)
    ds = ds.assign_coords({tdim: ds[tdim] + np.timedelta64(LOCAL_UTC_OFFSET_HOURS, "h")})
    ds_loaded = ds.load()
    daily = xr.Dataset()
    rs = ds_loaded.resample({tdim: "1D"})
    daily["wind_speed_max"] = rs.max()["wind_speed"]
    daily["wind_speed_mean"] = rs.mean()["wind_speed"]
    daily["gust_max"] = rs.max()[names["gust"]]
    daily["blh_max"] = rs.max()[names["blh"]]
    daily["t2m_max"] = rs.max()[names["t2m"]]
    daily["vpd_max"] = rs.max()["vpd"]
    daily["swvl1_mean"] = rs.mean()[names["swvl1"]]
    daily["precip_sum"] = rs.sum()[names["tp"]] * 1000.0
    daily["ssrd_mean"] = rs.sum()[names["ssrd"]] / 86400.0
    u, v = (ds_loaded[names["u10"]], ds_loaded[names["v10"]])
    u_bar = u.resample({tdim: "1D"}).mean()
    v_bar = v.resample({tdim: "1D"}).mean()
    daily["wind_dir_vec"] = (270.0 - np.degrees(np.arctan2(v_bar, u_bar))) % 360.0
    daily["precip_7d"] = daily["precip_sum"].rolling({tdim: 7}, min_periods=1).sum()
    daily["precip_30d"] = daily["precip_sum"].rolling({tdim: 30}, min_periods=1).sum()
    daily.attrs["time_zone"] = f"UTC+{LOCAL_UTC_OFFSET_HOURS} (Riyadh local)"
    return daily

def main():
    print("\n" + "=" * 60)
    print("STAGE 2 - DERIVED VARIABLES + DAILY AGGREGATION (Yearly Chunks)")
    all_files = sorted(RAW_DIR.glob("ERA5_*.nc"))
    years = sorted(list(set((int(f.name[5:9]) for f in all_files))))
    years = [y for y in years if y == 2025]
    all_regional_yearly_ds = []
    all_riyadh_yearly_dfs = []
    for year in years:
        print(f"\nProcessing year: {year}")
        yearly_files = [f for f in all_files if int(f.name[5:9]) == year]
        if not yearly_files:
            print(f"No files found for year {year}, skipping.")
            continue
        ds_year = xr.open_mfdataset(
            yearly_files, combine="by_coords", chunks={"valid_time": 24 * 31}, parallel=False
        )
        ds_year = drop_expver(ds_year)
        names, missing = resolve_names(ds_year)
        if missing:
            raise KeyError(f"Missing variables in merged dataset for {year}: {missing}")
        print(f"Resolved variable names for {year}: {names}")
        ds_year = add_derived(ds_year, names)
        daily_year = to_daily(ds_year, names)
        print(f"Computing daily fields for year {year}...")
        daily_year = daily_year.compute()
        out_nc_year = PROC_DIR / f"ERA5_daily_regional_{year}.nc"
        encoding = {v: {"zlib": True, "complevel": 4} for v in daily_year.data_vars}
        daily_year.to_netcdf(out_nc_year, encoding=encoding)
        print(f"Saved regional daily fields for {year} -> {out_nc_year}")
        all_regional_yearly_ds.append(daily_year)
        point_year = daily_year.sel(latitude=RIYADH_LAT, longitude=RIYADH_LON, method="nearest")
        df_year = point_year.to_dataframe().reset_index()
        out_csv_year = PROC_DIR / f"ERA5_daily_riyadh_{year}.csv"
        df_year.to_csv(out_csv_year, index=False)
        print(f"Saved Riyadh point series for {year} -> {out_csv_year}")
        all_riyadh_yearly_dfs.append(df_year)
    if all_regional_yearly_ds:
        print("\nConcatenating all yearly regional datasets...")
        full_regional_ds = xr.concat(all_regional_yearly_ds, dim="valid_time")
        out_nc_full = PROC_DIR / "ERA5_daily_regional_full.nc"
        encoding_full = {v: {"zlib": True, "complevel": 4} for v in full_regional_ds.data_vars}
        full_regional_ds.to_netcdf(out_nc_full, encoding=encoding_full)
        print(f"Saved full regional daily fields -> {out_nc_full}")
    if all_riyadh_yearly_dfs:
        print("\nConcatenating all yearly Riyadh point series...")
        full_riyadh_df = pd.concat(all_riyadh_yearly_dfs).sort_values("valid_time").reset_index(drop=True)
        out_csv_full = PROC_DIR / "ERA5_daily_riyadh_full.csv"
        full_riyadh_df.to_csv(out_csv_full, index=False)
        print(f"Saved full Riyadh point series -> {out_csv_full}")
        print("\nSummary of full daily receptor-point variables:")
        print(full_riyadh_df.describe().T[["count", "mean", "std", "min", "max"]])
    else:
        print("No data processed to concatenate.")
    print("\nProcessing complete.")

if __name__ == "__main__":
    main()

**1.4 ERA5 receptor series (24.75°N, 46.75°E)**

In [ ]:
old = PROC_DIR / "ERA5_daily_regional_full.nc"
if old.exists():
    old.unlink()
files = [f for f in sorted(PROC_DIR.glob("ERA5_daily_regional_*.nc")) if "full" not in f.name]
ds = xr.open_mfdataset(files, combine="by_coords")
TDIM = "valid_time" if "valid_time" in ds.dims else "time"
ds = ds.sortby(TDIM).compute()
point = ds.sel(latitude=24.71, longitude=46.68, method="nearest")
df = point.to_dataframe().reset_index()
agg = {
    "wind_speed_max": "max",
    "gust_max": "max",
    "blh_max": "max",
    "t2m_max": "max",
    "vpd_max": "max",
    "wind_speed_mean": "mean",
    "swvl1_mean": "mean",
    "ssrd_mean": "mean",
    "wind_dir_vec": "mean",
    "precip_sum": "sum",
}
agg = {k: v for k, v in agg.items() if k in df.columns}
df = df.groupby(TDIM, as_index=False).agg(agg).sort_values(TDIM)
df["precip_7d"] = df["precip_sum"].rolling(7, min_periods=1).sum()
df["precip_30d"] = df["precip_sum"].rolling(30, min_periods=1).sum()
df.to_csv(PROC_DIR / "ERA5_daily_riyadh.csv", index=False)
print(f"Rows: {len(df):,} (expect 4,018)")
print(f"End: {df[TDIM].max()} (expect 2025-12-31)")

**1.5 METAR download and dust catalogue**

In [ ]:
import io
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
OBS_DIR = PROJECT / "03_Observations" / "METAR"
RAW_DIR = OBS_DIR / "raw"
PROC_DIR = OBS_DIR / "processed"
for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)
ERA5_DAILY_CSV = PROJECT / "02_Meteorology" / "ERA5" / "processed" / "ERA5_daily_riyadh.csv"
STATION = "OERK"
NETWORK = "SA__ASOS"
START_YEAR = 2016
END_YEAR = 2025
LOCAL_UTC_OFFSET_HOURS = 3
REPORT_TYPES = (3,)
BASE_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
DUST_CODES = {
    "DU",
    "BLDU",
    "DS",
    "SA",
    "BLSA",
    "SS",
    "PO",
    "VCBLDU",
    "VCBLSA",
    "VCDS",
    "VCSS",
    "+DS",
    "+SS",
    "-DS",
    "-SS",
}
NON_DUST_OBSCURATION = {"HZ", "FU", "BR", "FG"}
VIS_MODERATE_KM = 5.0
VIS_STORM_KM = 1.0
MIN_PERSISTENCE_HOURS = 3
MILES_TO_KM = 1.60934

def build_url(year):
    params = [
        ("station", STATION),
        ("network", NETWORK),
        ("data", "vsby"),
        ("data", "wxcodes"),
        ("data", "drct"),
        ("data", "sknt"),
        ("data", "tmpf"),
        ("data", "dwpf"),
        ("year1", year),
        ("month1", 1),
        ("day1", 1),
        ("year2", year + 1),
        ("month2", 1),
        ("day2", 1),
        ("tz", "Etc/UTC"),
        ("format", "onlycomma"),
        ("latlon", "no"),
        ("missing", "M"),
        ("trace", "T"),
        ("direct", "no"),
    ]
    params += [("report_type", rt) for rt in REPORT_TYPES]
    return (BASE_URL, params)

def download_year(year):
    out = RAW_DIR / f"METAR_{STATION}_{year}.csv"
    if out.exists() and out.stat().st_size > 1024:
        print(f"  {year}  cached")
        return out
    url, params = build_url(year)
    for attempt in range(1, 4):
        try:
            resp = requests.get(url, params=params, timeout=300)
            resp.raise_for_status()
            text = resp.text
            if len(text) < 500 or "station" not in text[:200]:
                raise RuntimeError(f"Response too short or malformed: {text[:200]}")
            out.write_text(text)
            n_lines = text.count("\n")
            print(f"  {year}  downloaded  {n_lines:,} rows")
            return out
        except Exception as exc:
            print(f"  {year}  attempt {attempt}/3 failed: {str(exc)[:120]}")
            if attempt < 3:
                time.sleep(15 * attempt)
    raise RuntimeError(f"Could not download {year}")

def download_all():
    print(f"Downloading METAR for {STATION}, {START_YEAR}-{END_YEAR}")
    return [download_year(y) for y in range(START_YEAR, END_YEAR + 1)]

def parse_files(paths):
    frames = []
    for p in paths:
        df = pd.read_csv(p, na_values=["M", "T", ""], low_memory=False)
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df["valid"] = pd.to_datetime(df["valid"], utc=True, errors="coerce")
    df = df.dropna(subset=["valid"]).sort_values("valid")
    df["vis_km"] = pd.to_numeric(df["vsby"], errors="coerce") * MILES_TO_KM
    for col in ["drct", "sknt", "tmpf", "dwpf"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["wind_speed_ms"] = df["sknt"] * 0.514444
    df["wxcodes"] = df["wxcodes"].fillna("").astype(str)
    return df.reset_index(drop=True)

def flag_dust(df):
    tokens = df["wxcodes"].str.split()
    df["is_dust"] = tokens.apply(lambda t: bool(DUST_CODES.intersection(t)) if isinstance(t, list) else False)
    df["is_haze"] = tokens.apply(lambda t: "HZ" in t if isinstance(t, list) else False)
    df["dust_moderate"] = df["is_dust"] & (df["vis_km"] < VIS_MODERATE_KM)
    df["dust_storm"] = df["is_dust"] & (df["vis_km"] < VIS_STORM_KM)
    df["extinction_km"] = 3.912 / df["vis_km"].replace(0, np.nan)
    return df

def build_daily(df):
    local = df.copy()
    local["local_time"] = local["valid"] + pd.Timedelta(hours=LOCAL_UTC_OFFSET_HOURS)
    local["date"] = local["local_time"].dt.date
    local["hour"] = local["local_time"].dt.floor("h")
    hourly = (
        local.groupby("hour")
        .agg(
            is_dust=("is_dust", "any"),
            is_haze=("is_haze", "any"),
            dust_storm=("dust_storm", "any"),
            vis_km=("vis_km", "min"),
            extinction_km=("extinction_km", "max"),
            wind_speed_ms=("wind_speed_ms", "max"),
            drct=("drct", "mean"),
        )
        .reset_index()
    )
    hourly["date"] = hourly["hour"].dt.date
    daily = (
        hourly.groupby("date")
        .agg(
            n_obs_hours=("hour", "count"),
            dust_hours=("is_dust", "sum"),
            haze_hours=("is_haze", "sum"),
            storm_hours=("dust_storm", "sum"),
            vis_min_km=("vis_km", "min"),
            vis_mean_km=("vis_km", "mean"),
            extinction_max=("extinction_km", "max"),
            wind_max_ms=("wind_speed_ms", "max"),
        )
        .reset_index()
    )
    daily["date"] = pd.to_datetime(daily["date"])
    daily["dust_day"] = daily["dust_hours"] >= MIN_PERSISTENCE_HOURS
    daily["storm_day"] = daily["storm_hours"] >= 1
    daily["extreme_day"] = daily["dust_day"] & (daily["vis_min_km"] < VIS_STORM_KM)
    daily.loc[daily["n_obs_hours"] < 12, ["dust_day", "storm_day", "extreme_day"]] = np.nan
    return (daily, hourly)

def continuity_check(df, daily):
    df = df.copy()
    df["year"] = df["valid"].dt.year
    annual = (
        df.groupby("year")
        .agg(
            n_obs=("valid", "count"),
            n_with_wx=("wxcodes", lambda s: (s.str.len() > 0).sum()),
            dust_obs=("is_dust", "sum"),
            vis_reported=("vis_km", lambda s: s.notna().sum()),
        )
        .reset_index()
    )
    annual["wx_reporting_rate"] = annual["n_with_wx"] / annual["n_obs"]
    annual["dust_frequency"] = annual["dust_obs"] / annual["n_obs"]
    d = daily.copy()
    d["year"] = d["date"].dt.year
    annual_days = (
        d.groupby("year").agg(dust_days=("dust_day", "sum"), storm_days=("storm_day", "sum")).reset_index()
    )
    annual = annual.merge(annual_days, on="year", how="left")
    annual.to_csv(PROC_DIR / "METAR_annual_qc.csv", index=False)
    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
    axes[0].bar(annual["year"], annual["n_obs"], color="0.6")
    axes[0].set_ylabel("Observations per year")
    axes[0].set_title(f"{STATION} reporting continuity diagnostics")
    axes[1].plot(annual["year"], annual["wx_reporting_rate"], "o-", color="tab:orange")
    axes[1].set_ylabel("Fraction with\npresent-weather coded")
    axes[1].set_ylim(0, 1)
    axes[2].plot(annual["year"], annual["dust_days"], "o-", color="tab:red", label="Dust days")
    axes[2].plot(annual["year"], annual["storm_days"], "s--", color="tab:brown", label="Storm days")
    axes[2].set_ylabel("Days per year")
    axes[2].set_xlabel("Year")
    axes[2].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PROC_DIR / "METAR_continuity_check.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return annual

def main():
    print("STAGE 1 - DOWNLOAD")
    paths = download_all()
    print("\n" + "=" * 60)
    print("STAGE 2 - PARSE AND FLAG")
    df = parse_files(paths)
    df = flag_dust(df)
    df.to_csv(PROC_DIR / "METAR_hourly_riyadh.csv", index=False)
    print(f"Observations parsed : {len(df):,}")
    print(f"Period              : {df['valid'].min()} to {df['valid'].max()}")
    print(f"Dust observations   : {int(df['is_dust'].sum()):,} ({100 * df['is_dust'].mean():.2f}%)")
    print(
        f"Haze observations   : {int(df['is_haze'].sum()):,} ({100 * df['is_haze'].mean():.2f}%)  [excluded from events]"
    )
    print("\nMost frequent dust codes observed:")
    codes = df.loc[df["is_dust"], "wxcodes"].value_counts().head(10)
    for code, n in codes.items():
        print(f"  {code:12s} {n:6,}")
    print("\n" + "=" * 60)
    print("STAGE 3 - DAILY EVENT CATALOGUE")
    daily, hourly = build_daily(df)
    daily.to_csv(PROC_DIR / "METAR_daily_events.csv", index=False)
    n_days = len(daily)
    print(f"Days in catalogue   : {n_days:,}")
    print(f"Unclassifiable days : {int(daily['dust_day'].isna().sum()):,} (insufficient reports)")
    print(f"Dust days           : {int(daily['dust_day'].sum(skipna=True)):,}")
    print(f"Storm days          : {int(daily['storm_day'].sum(skipna=True)):,}")
    print(f"Extreme days        : {int(daily['extreme_day'].sum(skipna=True)):,}")
    print("\nSeasonal distribution of dust days:")
    seasonal = daily.loc[daily["dust_day"] == True].groupby(daily["date"].dt.month).size()
    for month, n in seasonal.items():
        print(f"  {month:02d}  {'#' * int(n / max(1, seasonal.max()) * 40)} {n}")
    print("\n" + "=" * 60)
    print("STAGE 4 - REPORTING CONTINUITY")
    annual = continuity_check(df, daily)
    print(annual[["year", "n_obs", "wx_reporting_rate", "dust_days", "storm_days"]].to_string(index=False))
    print("\nInspect METAR_continuity_check.png. A step change in the")
    print("middle panel indicates a reporting change, not a dust trend.")
    if ERA5_DAILY_CSV.exists():
        print("\n" + "=" * 60)
        print("STAGE 5 - JOIN TO ERA5 DAILY FIELDS")
        era5 = pd.read_csv(ERA5_DAILY_CSV)
        tcol = "valid_time" if "valid_time" in era5.columns else "time"
        era5[tcol] = pd.to_datetime(era5[tcol]).dt.tz_localize(None).dt.normalize()
        merged = daily.merge(era5, left_on="date", right_on=tcol, how="inner")
        merged.to_csv(PROC_DIR / "ANALYSIS_daily_merged.csv", index=False)
        print(f"Merged rows: {len(merged):,}")
        print(f"Saved -> {PROC_DIR / 'ANALYSIS_daily_merged.csv'}")
        print("\nThis table is the input to the emission-indicator step.")
    else:
        print("\nERA5 daily file not found; run Notebook 02B first to")
        print("produce the merged analysis table.")
    print("\n" + "=" * 60)
    print("DONE")

if __name__ == "__main__":
    main()

**1.6 Record homogeneity**

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
PROC_DIR = PROJECT / "03_Observations" / "METAR" / "processed"
HOURLY_CSV = PROC_DIR / "METAR_hourly_riyadh.csv"
NON_DUST_WEATHER = {"RA", "-RA", "+RA", "TS", "TSRA", "SHRA", "HZ", "BR", "FG"}
VIS_LOW_KM = 5.0
VIS_STORM_KM = 1.0

def load():
    df = pd.read_csv(HOURLY_CSV, low_memory=False)
    df["valid"] = pd.to_datetime(df["valid"], utc=True, errors="coerce")
    df = df.dropna(subset=["valid"])
    df["wxcodes"] = df["wxcodes"].fillna("").astype(str)
    df["year"] = df["valid"].dt.year
    return df

def annual_tests(df):
    tokens = df["wxcodes"].str.split()
    df = df.copy()
    df["has_nondust_wx"] = tokens.apply(
        lambda t: bool(NON_DUST_WEATHER.intersection(t)) if isinstance(t, list) else False
    )
    df["any_wx"] = df["wxcodes"].str.len() > 0
    df["vis_low"] = df["vis_km"] < VIS_LOW_KM
    df["vis_storm"] = df["vis_km"] < VIS_STORM_KM
    annual = (
        df.groupby("year")
        .agg(
            n_obs=("valid", "count"),
            vis_reported=("vis_km", lambda s: s.notna().sum()),
            nondust_wx_rate=("has_nondust_wx", "mean"),
            any_wx_rate=("any_wx", "mean"),
            dust_rate=("is_dust", "mean"),
            vis_low_hours=("vis_low", "sum"),
            vis_storm_hours=("vis_storm", "sum"),
            vis_median=("vis_km", "median"),
            vis_p05=("vis_km", lambda s: s.quantile(0.05)),
        )
        .reset_index()
    )
    coded = df[df["vis_low"]].groupby("year")["is_dust"].mean().rename("frac_lowvis_coded_dust").reset_index()
    annual = annual.merge(coded, on="year", how="left")
    return annual

def plot(annual):
    fig, axes = plt.subplots(4, 1, figsize=(9, 12), sharex=True)
    axes[0].plot(annual["year"], annual["dust_rate"], "o-", color="tab:red", label="Dust codes")
    axes[0].plot(
        annual["year"],
        annual["nondust_wx_rate"],
        "s--",
        color="tab:blue",
        label="Non-dust codes (rain/TS/haze)",
    )
    axes[0].set_ylabel("Fraction of observations")
    axes[0].set_title("TEST 1: do unrelated weather codes fall at the same time?")
    axes[0].legend(fontsize=8)
    axes[1].plot(
        annual["year"],
        annual["vis_low_hours"],
        "o-",
        color="tab:green",
        label=f"Hours vis < {VIS_LOW_KM:g} km",
    )
    axes[1].plot(
        annual["year"],
        annual["vis_storm_hours"],
        "s--",
        color="darkgreen",
        label=f"Hours vis < {VIS_STORM_KM:g} km",
    )
    axes[1].set_ylabel("Hours per year")
    axes[1].set_title("TEST 2: does the instrument record show the same decline?")
    axes[1].legend(fontsize=8)
    axes[2].plot(annual["year"], annual["vis_median"], "o-", color="tab:purple", label="Median")
    axes[2].plot(annual["year"], annual["vis_p05"], "s--", color="indigo", label="5th percentile")
    axes[2].set_ylabel("Visibility (km)")
    axes[2].set_title("Visibility distribution")
    axes[2].legend(fontsize=8)
    axes[3].plot(annual["year"], annual["frac_lowvis_coded_dust"], "o-", color="tab:orange")
    axes[3].set_ylabel("Fraction")
    axes[3].set_xlabel("Year")
    axes[3].set_title(
        "Share of low-visibility hours carrying a dust code\n(a fall here with flat visibility = coding change)"
    )
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PROC_DIR / "HOMOGENEITY_diagnostic.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def verdict(annual):
    early = annual[annual["year"] <= 2022]
    late = annual[annual["year"] >= 2023]
    if late.empty or early.empty:
        print("Not enough years to compare.")
        return

    def pct_change(col):
        e, l = (early[col].mean(), late[col].mean())
        if e == 0 or np.isnan(e):
            return np.nan
        return 100 * (l - e) / e

    print("\n" + "=" * 60)
    print("VERDICT: 2023-2024 vs 2015-2022")
    rows = [
        ("Dust code rate", "dust_rate"),
        ("Non-dust code rate", "nondust_wx_rate"),
        ("Hours vis < 5 km", "vis_low_hours"),
        ("Hours vis < 1 km", "vis_storm_hours"),
        ("Median visibility", "vis_median"),
        ("Low-vis hours coded dust", "frac_lowvis_coded_dust"),
    ]
    for label, col in rows:
        print(f"  {label:28s} {pct_change(col):+8.1f}%")
    dust_drop = pct_change("dust_rate")
    vis_drop = pct_change("vis_low_hours")
    print("\nInterpretation:")
    if dust_drop < -50 and (np.isnan(vis_drop) or vis_drop > -25):
        print("  REPORTING ARTEFACT. Dust coding collapsed while the")
        print("  visibility record did not. The dust was still occurring.")
        print("  Do not treat 2023-2024 as low-dust years.")
    elif dust_drop < -50 and vis_drop < -50:
        print("  CONSISTENT DECLINE. Both the coded and instrument")
        print("  records fall together, so the drop may be physical.")
        print("  Check regional precipitation and vegetation before")
        print("  accepting it as a real trend.")
    else:
        print("  AMBIGUOUS. Inspect the figure panel by panel.")

def main():
    df = load()
    annual = annual_tests(df)
    annual.to_csv(PROC_DIR / "HOMOGENEITY_annual.csv", index=False)
    cols = [
        "year",
        "dust_rate",
        "nondust_wx_rate",
        "vis_low_hours",
        "vis_storm_hours",
        "vis_median",
        "frac_lowvis_coded_dust",
    ]
    print(annual[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    plot(annual)
    verdict(annual)
    print(f"\nFigure -> {PROC_DIR / 'HOMOGENEITY_diagnostic.png'}")

if __name__ == "__main__":
    main()

**1.7 Locally raised / “suspension-dominated” daily classes**

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
METAR_PROC = PROJECT / "03_Observations" / "METAR" / "processed"
ERA5_PROC = PROJECT / "02_Meteorology" / "ERA5" / "processed"
OUT_DIR = PROJECT / "04_Analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HOURLY_CSV = METAR_PROC / "METAR_hourly_riyadh.csv"
ERA5_DAILY_CSV = ERA5_PROC / "ERA5_daily_riyadh.csv"
LOCAL_UTC_OFFSET_HOURS = 3
MIN_PERSISTENCE_HOURS = 3
LOCAL_CODES = {"BLDU", "BLSA"}
TRANSPORT_CODES = {"DU", "SA"}
SEGMENTS = [("Baseline", 2016, 2022), ("Minimum", 2023, 2024), ("Recovery", 2025, 2025)]
DUST_SEASON = [3, 4, 5, 6]
DRIVERS = [
    "wind_speed_max",
    "gust_max",
    "swvl1_mean",
    "precip_sum",
    "precip_7d",
    "precip_30d",
    "vpd_max",
    "blh_max",
    "t2m_max",
    "ssrd_mean",
]

def classify(df):
    tokens = df["wxcodes"].fillna("").astype(str).str.split()
    tokens = tokens.apply(lambda t: set(t) if isinstance(t, list) else set())
    df = df.copy()
    df["is_local"] = tokens.apply(lambda t: bool(LOCAL_CODES & t))
    df["is_transport"] = tokens.apply(lambda t: bool(TRANSPORT_CODES & t) and (not bool(LOCAL_CODES & t)))
    df["is_stormcode"] = tokens.apply(
        lambda t: bool({"DS", "SS"} & t) and (not bool(LOCAL_CODES & t)) and (not bool(TRANSPORT_CODES & t))
    )
    return df

def to_daily(df):
    df = df.copy()
    df["local_time"] = df["valid"] + pd.Timedelta(hours=LOCAL_UTC_OFFSET_HOURS)
    df["hour"] = df["local_time"].dt.floor("h")
    hourly = (
        df.groupby("hour")
        .agg(
            is_local=("is_local", "any"),
            is_transport=("is_transport", "any"),
            is_stormcode=("is_stormcode", "any"),
            is_dust=("is_dust", "any"),
            vis_km=("vis_km", "min"),
            extinction_km=("extinction_km", "max"),
        )
        .reset_index()
    )
    hourly["date"] = hourly["hour"].dt.normalize()
    daily = (
        hourly.groupby("date")
        .agg(
            n_obs_hours=("hour", "count"),
            local_hours=("is_local", "sum"),
            transport_hours=("is_transport", "sum"),
            storm_hours=("is_stormcode", "sum"),
            dust_hours=("is_dust", "sum"),
            vis_min_km=("vis_km", "min"),
            extinction_max=("extinction_km", "max"),
        )
        .reset_index()
    )
    daily["local_day"] = (daily["local_hours"] >= MIN_PERSISTENCE_HOURS).astype(float)
    daily["transport_day"] = (daily["transport_hours"] >= MIN_PERSISTENCE_HOURS).astype(float)
    daily["dust_day"] = (daily["dust_hours"] >= MIN_PERSISTENCE_HOURS).astype(float)
    insufficient = daily["n_obs_hours"] < 12
    daily.loc[insufficient, ["local_day", "transport_day", "dust_day"]] = np.nan
    return daily

def merge_era5(daily):
    if not ERA5_DAILY_CSV.exists():
        raise FileNotFoundError(f"{ERA5_DAILY_CSV} not found.")
    era5 = pd.read_csv(ERA5_DAILY_CSV, low_memory=False)
    tcol = "valid_time" if "valid_time" in era5.columns else "time"
    era5[tcol] = pd.to_datetime(era5[tcol]).dt.tz_localize(None).dt.normalize()
    daily["date"] = pd.to_datetime(daily["date"]).dt.tz_localize(None).dt.normalize()
    merged = daily.merge(era5, left_on="date", right_on=tcol, how="inner")
    merged["year"] = merged["date"].dt.year
    merged["month"] = merged["date"].dt.month
    return merged

def add_anomalies(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[f"{c}_anom"] = df[c] - df.groupby("month")[c].transform("mean")
    return df

def correlate(df, target, drivers, label):
    valid = df.dropna(subset=[target])
    rows = []
    for d in drivers:
        if d not in valid.columns:
            continue
        sub = valid.dropna(subset=[d])
        if len(sub) < 100 or sub[d].nunique() < 5:
            continue
        rho, p = stats.spearmanr(sub[d], sub[target])
        rows.append({"class": label, "driver": d, "rho": rho, "p": p, "n": len(sub)})
    return pd.DataFrame(rows)

def report_correlations(df):
    anom_drivers = [f"{d}_anom" for d in DRIVERS]
    df = add_anomalies(df, DRIVERS)
    frames = []
    for target, label in [
        ("local_day", "LOCAL (BLDU/BLSA)"),
        ("transport_day", "TRANSPORT (DU/SA)"),
        ("dust_day", "ALL DUST"),
    ]:
        frames.append(correlate(df, target, DRIVERS, label + " raw"))
        frames.append(correlate(df, target, anom_drivers, label + " anom"))
    out = pd.concat(frames, ignore_index=True)
    print("\n" + "=" * 78)
    print("DRIVER CORRELATIONS BY EVENT CLASS")
    for label in out["class"].unique():
        sub = out[out["class"] == label].sort_values("rho", key=abs, ascending=False).head(6)
        print(f"\n{label}")
        print(f"  {'driver':22s} {'rho':>8s} {'p':>11s} {'n':>7s}")
        for _, r in sub.iterrows():
            star = "*" if r["p"] < 0.001 else " "
            print(f"  {r['driver']:22s} {r['rho']:8.3f} {r['p']:11.2e}{star} {r['n']:7.0f}")
    print("\n" + "-" * 78)
    print("WIND RESPONSE: local vs transported (anomalies)")
    for drv in ["wind_speed_max_anom", "gust_max_anom"]:
        loc = out[(out["class"] == "LOCAL (BLDU/BLSA) anom") & (out["driver"] == drv)]
        tra = out[(out["class"] == "TRANSPORT (DU/SA) anom") & (out["driver"] == drv)]
        if loc.empty or tra.empty:
            continue
        rl, rt = (float(loc["rho"].iloc[0]), float(tra["rho"].iloc[0]))
        print(f"  {drv:22s} local {rl:+.3f}   transport {rt:+.3f}   ratio {(rl / rt if rt else np.nan):5.2f}")
        if rl > rt * 1.5:
            print("     -> local wind explains locally raised dust far better.")
            print("        Consistent with transport control of DU days.")
        elif abs(rl - rt) < 0.05:
            print("     -> both classes respond alike. Transport control")
            print("        is NOT supported; look elsewhere for 2024.")
    return out

def annual_breakdown(df):
    annual = (
        df.groupby("year")
        .agg(
            n_days=("date", "count"),
            local_days=("local_day", "sum"),
            transport_days=("transport_day", "sum"),
            dust_days=("dust_day", "sum"),
            wind_max_mean=("wind_speed_max", "mean"),
            swvl1_mean=("swvl1_mean", "mean"),
        )
        .reset_index()
    )

    def seg(y):
        for name, a, b in SEGMENTS:
            if a <= y <= b:
                return name
        return "Other"

    annual["segment"] = annual["year"].map(seg)
    print("\n" + "=" * 78)
    print("EVENT CLASSES BY YEAR")
    print(
        annual[
            ["year", "segment", "dust_days", "local_days", "transport_days", "wind_max_mean", "swvl1_mean"]
        ].to_string(index=False, float_format=lambda x: f"{x:.3f}")
    )
    seg_means = annual.groupby("segment")[["local_days", "transport_days", "dust_days"]].mean()
    if "Baseline" in seg_means.index:
        print("\n" + "-" * 78)
        print("CHANGE FROM BASELINE (mean days per year)")
        base = seg_means.loc["Baseline"]
        for name, _, _ in SEGMENTS:
            if name == "Baseline" or name not in seg_means.index:
                continue
            row = seg_means.loc[name]
            print(f"  {name:10s}", end="")
            for col in ["local_days", "transport_days"]:
                pct = 100 * (row[col] - base[col]) / base[col] if base[col] else np.nan
                print(f"  {col.replace('_days', ''):10s} {row[col]:6.1f} ({pct:+6.1f}%)", end="")
        print("\n  If transported days fell much further than local days,")
        print("  the decline originated upwind, not at the receptor.")
    return annual

def plot(annual, df):
    fig, axes = plt.subplots(3, 1, figsize=(9, 10))
    w = 0.38
    x = np.arange(len(annual))
    axes[0].bar(x - w / 2, annual["transport_days"], w, label="Transported (DU/SA)", color="tab:blue")
    axes[0].bar(x + w / 2, annual["local_days"], w, label="Locally raised (BLDU/BLSA)", color="tab:orange")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(annual["year"], rotation=45)
    axes[0].set_ylabel("Days per year")
    axes[0].set_title("Dust days by emission class")
    axes[0].legend(fontsize=8)
    share = annual["transport_days"] / (annual["transport_days"] + annual["local_days"]).replace(0, np.nan)
    axes[1].plot(annual["year"], share, "o-", color="tab:purple")
    axes[1].axhline(0.5, color="0.7", ls=":")
    axes[1].set_ylabel("Transported share")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Fraction of dust days that were transported")
    for name, a, b in SEGMENTS:
        if name == "Minimum":
            axes[1].axvspan(a - 0.5, b + 0.5, color="red", alpha=0.06)
    bins = np.arange(0, 16, 1.5)
    df = df.copy()
    df["wind_bin"] = pd.cut(df["wind_speed_max"], bins)
    grouped = (
        df.groupby("wind_bin", observed=True)
        .agg(local=("local_day", "mean"), transport=("transport_day", "mean"), n=("local_day", "size"))
        .reset_index()
    )
    grouped = grouped[grouped["n"] >= 20]
    centres = [iv.mid for iv in grouped["wind_bin"]]
    axes[2].plot(centres, grouped["local"], "o-", color="tab:orange", label="Locally raised")
    axes[2].plot(centres, grouped["transport"], "s-", color="tab:blue", label="Transported")
    axes[2].set_xlabel("Daily maximum 10 m wind speed (m s-1)")
    axes[2].set_ylabel("Probability of event")
    axes[2].set_title("Response to LOCAL wind speed")
    axes[2].legend(fontsize=8)
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "CLASS_comparison.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"\nFigure -> {OUT_DIR / 'CLASS_comparison.png'}")

def main():
    print("NOTEBOOK 05 - TRANSPORT VS LOCAL DUST")
    hourly = pd.read_csv(HOURLY_CSV, low_memory=False)
    hourly["valid"] = pd.to_datetime(hourly["valid"], utc=True, errors="coerce")
    hourly = hourly.dropna(subset=["valid"])
    hourly = classify(hourly)
    n_dust = int(hourly["is_dust"].sum())
    print(f"\nHourly observations : {len(hourly):,}")
    print(f"Dust observations   : {n_dust:,}")
    print(f"  locally raised    : {int(hourly['is_local'].sum()):,}")
    print(f"  transported       : {int(hourly['is_transport'].sum()):,}")
    print(f"  storm code only   : {int(hourly['is_stormcode'].sum()):,}")
    daily = to_daily(hourly)
    df = merge_era5(daily)
    df.to_csv(OUT_DIR / "CLASS_daily_events.csv", index=False)
    print(f"\nMerged days         : {len(df):,}")
    print(f"  local days        : {int(df['local_day'].sum()):,}")
    print(f"  transport days    : {int(df['transport_day'].sum()):,}")
    print(f"  all dust days     : {int(df['dust_day'].sum()):,}")
    annual = annual_breakdown(df)
    annual.to_csv(OUT_DIR / "CLASS_annual_summary.csv", index=False)
    corr = report_correlations(df)
    corr.to_csv(OUT_DIR / "CLASS_correlations.csv", index=False)
    plot(annual, df)
    print("\n" + "=" * 78)
    print("HOW TO READ THIS")
    print("  Transport control is supported if BOTH hold:")
    print("    1. local wind predicts BLDU days much better than DU days")
    print("    2. transported days fell further than local days in 2023-24")
    print("  If supported, the emission indicator belongs at the SOURCE")
    print("  cells along the trajectory paths, not at the receptor, and")
    print("  the ERA5 domain must be extended north past 32N to cover")
    print("  the Fertile Crescent sources.")
    print("  If not supported, the 2024 anomaly needs another")
    print("  explanation before the indicator is built.")

if __name__ == "__main__":
    main()

**1.8 HYSPLIT backward trajectories (2016–2025, 4 per day)**

In [ ]:
import os
import shutil
import subprocess
import tarfile
import time
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import requests

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
ANALYSIS_DIR = PROJECT / "04_Analysis"
EVENTS_CSV = ANALYSIS_DIR / "CLASS_daily_events.csv"
HYSPLIT_DIR = PROJECT / "05_HYSPLIT"
TRAJ_DIR = HYSPLIT_DIR / "trajectories"
MANIFEST_CSV = HYSPLIT_DIR / "TRAJ_manifest.csv"
HYSPLIT_TARBALL = HYSPLIT_DIR / "hysplit_linux.tar.gz"
LOCAL_ROOT = Path("/content/hysplit_work")
MET_DIR = LOCAL_ROOT / "met"
RUN_DIR = LOCAL_ROOT / "run"
EXEC_DIR = LOCAL_ROOT / "hysplit" / "exec"
for d in (HYSPLIT_DIR, TRAJ_DIR, LOCAL_ROOT, MET_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
RECEPTOR_LAT = 24.9578
RECEPTOR_LON = 46.6989
ARRIVAL_HEIGHT_M = 500.0
RUN_HOURS = -84
START_HOURS = ["00", "06", "12", "18"]
VERTICAL_MOTION = 0
MODEL_TOP_M = 10000.0
EVENT_COLUMN = "dust_day"
MET_URL_PATTERNS = [
    "https://noaa-oar-arl-hysplit-pds.s3.amazonaws.com/gdas1/{fname}",
    "https://www.ready.noaa.gov/data/archives/gdas1/{fname}",
    "ftp://arlftp.arlhq.noaa.gov/pub/archives/gdas1/{fname}",
]
MONTH_ABBR = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
DOWNLOAD_TIMEOUT = 1800
DOWNLOAD_RETRIES = 3

def install_hysplit():
    hyts = EXEC_DIR / "hyts_std"
    if hyts.exists() and os.access(hyts, os.X_OK):
        print(f"HYSPLIT already present: {hyts}")
        return hyts
    if not HYSPLIT_TARBALL.exists():
        raise FileNotFoundError(
            f"\n{HYSPLIT_TARBALL} not found.\n\nHYSPLIT for Linux is behind a registration form and\ncannot be fetched automatically. Register and download\nthe tarball here:\n\n    https://www.ready.noaa.gov/HYSPLIT_linuxtrial.php\n\nthen place it at the path above and re-run.\n"
        )
    print(f"Unpacking {HYSPLIT_TARBALL.name} ...")
    dest = LOCAL_ROOT / "hysplit"
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(HYSPLIT_TARBALL) as tf:
        tf.extractall(dest)
    found = list(dest.rglob("hyts_std"))
    if not found:
        raise RuntimeError(
            "hyts_std not found in the tarball. Check that the download is the Linux distribution, not Windows."
        )
    hyts = found[0]
    exec_dir = hyts.parent
    for f in exec_dir.iterdir():
        if f.is_file():
            f.chmod(0o755)
    print(f"HYSPLIT installed: {hyts}")
    ascdata = list(dest.rglob("ASCDATA.CFG"))
    if ascdata:
        shutil.copy2(ascdata[0], RUN_DIR / "ASCDATA.CFG")
        print("ASCDATA.CFG staged in the run directory.")
    return hyts

def met_filename(dt):
    week = min((dt.day - 1) // 7 + 1, 5)
    return f"gdas1.{MONTH_ABBR[dt.month - 1]}{dt.year % 100:02d}.w{week}"

def required_met_files(start_dt, run_hours):
    files = []
    step = timedelta(hours=6)
    t = start_dt
    end = start_dt + timedelta(hours=run_hours)
    while t >= end:
        f = met_filename(t)
        if f not in files:
            files.append(f)
        t -= step
    f = met_filename(end)
    if f not in files:
        files.append(f)
    return files

def build_runlist(df):
    events = events = df.dropna(subset=[EVENT_COLUMN]).copy()
    events["date"] = pd.to_datetime(events["date"])
    runs = []
    for _, row in events.iterrows():
        for hh in START_HOURS:
            start = datetime(row["date"].year, row["date"].month, row["date"].day, int(hh))
            runs.append(
                {
                    "start": start,
                    "tag": start.strftime("%Y%m%d_%H"),
                    "primary_met": met_filename(start),
                    "met_files": required_met_files(start, RUN_HOURS),
                    "local_day": row.get("local_day"),
                    "transport_day": row.get("transport_day"),
                }
            )
    runs = pd.DataFrame(runs)
    runs = runs.sort_values("start").reset_index(drop=True)
    return runs

def download_met(fname):
    dest = MET_DIR / fname
    if dest.exists() and dest.stat().st_size > 100 * 1024**2:
        return dest
    for pattern in MET_URL_PATTERNS:
        url = pattern.format(fname=fname)
        if url.startswith("ftp://"):
            continue
        for attempt in range(1, DOWNLOAD_RETRIES + 1):
            try:
                print(f"    downloading {fname} ... ", end="", flush=True)
                t0 = time.time()
                with requests.get(url, stream=True, timeout=DOWNLOAD_TIMEOUT) as r:
                    if r.status_code != 200:
                        print(f"HTTP {r.status_code}")
                        break
                    tmp = dest.with_suffix(".part")
                    with open(tmp, "wb") as fh:
                        for chunk in r.iter_content(chunk_size=8 * 1024**2):
                            fh.write(chunk)
                    tmp.rename(dest)
                mb = dest.stat().st_size / 1024**2
                secs = time.time() - t0
                print(f"{mb:.0f} MB in {secs:.0f}s ({mb / secs:.1f} MB/s)")
                return dest
            except Exception as exc:
                print(f"failed ({str(exc)[:60]})")
                dest.with_suffix(".part").unlink(missing_ok=True)
                if attempt < DOWNLOAD_RETRIES:
                    time.sleep(10 * attempt)
    return None

def purge_met(keep):
    keep = set(keep)
    for f in MET_DIR.glob("gdas1.*"):
        if f.name not in keep:
            f.unlink(missing_ok=True)

def write_control(run, met_present, out_name):
    s = run["start"]
    lines = [
        f"{s.year % 100:02d} {s.month:02d} {s.day:02d} {s.hour:02d}",
        "1",
        f"{RECEPTOR_LAT:.4f} {RECEPTOR_LON:.4f} {ARRIVAL_HEIGHT_M:.1f}",
        f"{RUN_HOURS}",
        f"{VERTICAL_MOTION}",
        f"{MODEL_TOP_M:.1f}",
        f"{len(met_present)}",
    ]
    for m in met_present:
        lines.append(f"{MET_DIR}/")
        lines.append(m)
    lines.append(f"{TRAJ_DIR}/")
    lines.append(out_name)
    (RUN_DIR / "CONTROL").write_text("\n".join(lines) + "\n")

def run_one(hyts, run):
    out_name = f"tdump_{run['tag']}"
    out_path = TRAJ_DIR / out_name
    if out_path.exists() and out_path.stat().st_size > 1000:
        return "cached"
    met_present = [m for m in run["met_files"] if (MET_DIR / m).exists()]
    if not met_present:
        return "no_met"
    write_control(run, met_present, out_name)
    try:
        proc = subprocess.run([str(hyts)], cwd=RUN_DIR, capture_output=True, text=True, timeout=300)
    except subprocess.TimeoutExpired:
        return "timeout"
    if out_path.exists() and out_path.stat().st_size > 1000:
        return "ok"
    err = (proc.stderr or proc.stdout or "")[:120].replace("\n", " ")
    return f"failed: {err}"

def main():
    print("NOTEBOOK 07 - HYSPLIT TRAJECTORY GENERATION")
    hyts = install_hysplit()
    df = pd.read_csv(EVENTS_CSV, low_memory=False)
    runs = build_runlist(df)
    n_events = runs["tag"].str[:8].nunique()
    weeks = sorted(runs["primary_met"].unique())
    print(f"\nEvent days        : {n_events}")
    print(f"Start times/day   : {len(START_HOURS)}")
    print(f"Trajectories      : {len(runs):,}")
    print(f"Distinct met files: {len(weeks)}")
    print(f"Download volume   : ~{len(weeks) * 0.6:.0f} GB (streamed, not retained)")
    print(f"Peak local storage: ~1.2 GB")
    done = {f.name for f in TRAJ_DIR.glob("tdump_*")}
    print(f"Already complete  : {len(done):,}")
    grouped = runs.groupby("primary_met", sort=False)
    records = []
    t_start = time.time()
    for i, (met, group) in enumerate(grouped, start=1):
        pending = [r for _, r in group.iterrows() if f"tdump_{r['tag']}" not in done]
        if not pending:
            print(f"[{i}/{len(weeks)}] {met}: all {len(group)} cached")
            continue
        print(f"\n[{i}/{len(weeks)}] {met}: {len(pending)} runs pending")
        needed = sorted({m for r in pending for m in r["met_files"]})
        purge_met(needed)
        for m in needed:
            if download_met(m) is None:
                print(f"    WARNING: {m} unavailable, runs may be partial")
        for r in pending:
            status = run_one(hyts, r)
            records.append(
                {
                    "tag": r["tag"],
                    "start": r["start"],
                    "primary_met": r["primary_met"],
                    "local_day": r["local_day"],
                    "transport_day": r["transport_day"],
                    "status": status,
                }
            )
        ok = sum((1 for x in records[-len(pending) :] if x["status"] == "ok"))
        print(f"    {ok}/{len(pending)} succeeded")
        if records:
            pd.DataFrame(records).to_csv(
                MANIFEST_CSV, mode="a", header=not MANIFEST_CSV.exists(), index=False
            )
            records = []
    purge_met([])
    total = len(list(TRAJ_DIR.glob("tdump_*")))
    elapsed = (time.time() - t_start) / 60
    print("\n" + "=" * 78)
    print("SUMMARY")
    print(f"Trajectory files on Drive : {total:,}")
    print(f"Expected                  : {len(runs):,}")
    print(f"Runtime this session      : {elapsed:.0f} min")
    if total < len(runs):
        print(f"\n{len(runs) - total:,} remaining. Re-run this cell;")
        print("completed trajectories are skipped automatically.")
    else:
        print("\nComplete. Next: Notebook 08 for PSCF and CWT.")
        print("Run the primary analysis on 2016-2022 only, and treat")
        print("2023-2025 as a separate low-count sensitivity check.")

if __name__ == "__main__":
    main()

**1.9 Indicator models (logistic regression, random forest)**

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT = Path("/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
ANALYSIS = PROJECT / "04_Analysis"
IN_CSV = ANALYSIS / "CLASS_daily_events.csv"
SWEEP_CSV = ANALYSIS / "WINDAXIS_sweep.csv"
OUT = ANALYSIS
MAX_YEAR = 2025
TRAIN_YEARS = (2016, 2022)
TEST_YEARS = (2023, 2025)
INCLUDE_GUST = False
WIND_AXIS_MODE = "auto"
LITERATURE_AXIS_DEG = 320.0
AXIS_SOURCE_TARGET = "dust_day"
AXIS_AGREEMENT_DEG = 45.0
TARGETS = [
    ("local_day", "LOCAL (BLDU/BLSA)"),
    ("transport_day", "TRANSPORT (DU/SA)"),
    ("dust_day", "ALL DUST"),
]
BASE_DRIVERS = [
    "wind_speed_max",
    "swvl1_mean",
    "precip_sum",
    "precip_7d",
    "precip_30d",
    "vpd_max",
    "blh_max",
    "t2m_max",
    "ssrd_mean",
]
RANDOM_STATE = 0
N_TREES = 400
VIF_WARN = 5.0

def load():
    if not IN_CSV.exists():
        raise FileNotFoundError(f"{IN_CSV} not found. Run Notebook 05.")
    df = pd.read_csv(IN_CSV, low_memory=False)
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["doy"] = df["date"].dt.dayofyear
    before = len(df)
    df = df[df["year"] <= MAX_YEAR].copy()
    if len(df) < before:
        print(f"Clipped {before - len(df)} row(s) after {MAX_YEAR} (local-time boundary artifact)")
    for c in ["local_day", "transport_day", "dust_day"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    print(f"Rows: {len(df):,}  ({df['date'].min().date()} to {df['date'].max().date()})")
    return df

def resolve_axis():
    if isinstance(WIND_AXIS_MODE, (int, float)):
        return (float(WIND_AXIS_MODE), "set explicitly in configuration")
    if WIND_AXIS_MODE == "literature":
        return (LITERATURE_AXIS_DEG, "literature value, forced")
    if not SWEEP_CSV.exists():
        print(
            f"\n  {SWEEP_CSV.name} not found. Falling back to the literature axis ({LITERATURE_AXIS_DEG:.0f} deg)."
        )
        print("  Run Notebook 09A to set this empirically.")
        return (LITERATURE_AXIS_DEG, "literature value, sweep unavailable")
    sw = pd.read_csv(SWEEP_CSV)
    sub = sw[sw["target"] == AXIS_SOURCE_TARGET]
    if sub.empty:
        return (LITERATURE_AXIS_DEG, "literature value, target missing")
    best = float(sub.loc[sub["rho"].idxmax(), "axis_deg"])
    sep = abs(best - LITERATURE_AXIS_DEG)
    sep = min(sep, 360 - sep)
    if sep <= AXIS_AGREEMENT_DEG:
        return (
            LITERATURE_AXIS_DEG,
            f"literature value retained; empirical optimum {best:.0f} deg agrees within {sep:.0f} deg",
        )
    return (best, f"empirical optimum from Notebook 09A ({sep:.0f} deg from the literature value)")

def engineer(df, axis_deg):
    d = df.copy()
    theta = np.radians(d["wind_dir_vec"])
    d["wind_dir_sin"] = np.sin(theta)
    d["wind_dir_cos"] = np.cos(theta)
    d["wind_projection"] = d["wind_speed_max"] * np.cos(theta - np.radians(axis_deg))
    d["doy_sin"] = np.sin(2 * np.pi * d["doy"] / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * d["doy"] / 365.25)
    feats = list(BASE_DRIVERS)
    if INCLUDE_GUST and "gust_max" in d.columns:
        feats.insert(1, "gust_max")
    feats += ["wind_dir_sin", "wind_dir_cos", "wind_projection", "doy_sin", "doy_cos"]
    feats = [f for f in feats if f in d.columns]
    return (d, feats)

def check_vif(df, feats):
    d = df[feats].dropna()
    rows = []
    for c in feats:
        others = [o for o in feats if o != c]
        X = np.column_stack([np.ones(len(d)), d[others].values])
        y = d[c].values
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        ss_tot = ((y - y.mean()) ** 2).sum()
        r2 = 1 - (resid**2).sum() / ss_tot if ss_tot else np.nan
        rows.append({"feature": c, "vif": 1 / (1 - r2) if r2 < 1 else np.inf})
    vif = pd.DataFrame(rows).sort_values("vif", ascending=False)
    bad = vif[vif["vif"] > VIF_WARN]
    print(f"\n  Variance inflation (warn above {VIF_WARN:.0f}):")
    for _, r in vif.head(5).iterrows():
        flag = "  <-- unstable" if r["vif"] > VIF_WARN else ""
        print(f"    {r['feature']:20s} {r['vif']:8.2f}{flag}")
    if not bad.empty:
        print(f"\n  WARNING: {len(bad)} feature(s) above the threshold.")
        print("  Coefficient SIGNS for these are not interpretable.")
        print("  Report permutation importance and marginal association")
        print("  instead, or drop the redundant term.")
    return vif

def marginal_rho(df, feats, target):
    out = {}
    d = df.dropna(subset=[target])
    for f in feats:
        s = d.dropna(subset=[f])
        if len(s) < 100 or s[f].nunique() < 5:
            out[f] = np.nan
            continue
        out[f] = stats.spearmanr(s[f], s[target])[0]
    return out

def fit_one(df, feats, target, label):
    d = df.dropna(subset=[target] + feats)
    tr = d[(d["year"] >= TRAIN_YEARS[0]) & (d["year"] <= TRAIN_YEARS[1])]
    te = d[(d["year"] >= TEST_YEARS[0]) & (d["year"] <= TEST_YEARS[1])]
    n_tr, n_te = (int(tr[target].sum()), int(te[target].sum()))
    print("\n" + "=" * 70)
    print(f"{label}   target = {target}")
    print(
        f"  train {TRAIN_YEARS[0]}-{TRAIN_YEARS[1]}: {len(tr):,} days, {n_tr} events ({100 * tr[target].mean():.1f}%)"
    )
    print(
        f"  test  {TEST_YEARS[0]}-{TEST_YEARS[1]}: {len(te):,} days, {n_te} events ({100 * te[target].mean():.1f}%)"
    )
    if n_tr < 30:
        print("  SKIPPED: too few training events.")
        return (None, None, None)
    if n_te < 10:
        print(f"  WARNING: only {n_te} events in the test period; test metrics are unstable.")
    Xtr, ytr = (tr[feats].values, tr[target].values)
    Xte, yte = (te[feats].values, te[target].values)
    sc = StandardScaler().fit(Xtr)
    logit = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    logit.fit(sc.transform(Xtr), ytr)
    rf = RandomForestClassifier(
        n_estimators=N_TREES,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(Xtr, ytr)
    perf = []
    for name, model, A, B in [("logit", logit, sc.transform(Xtr), sc.transform(Xte)), ("rf", rf, Xtr, Xte)]:
        ptr, pte = (model.predict_proba(A)[:, 1], model.predict_proba(B)[:, 1])
        perf.append(
            {
                "class": label,
                "target": target,
                "model": name,
                "auc_train": roc_auc_score(ytr, ptr),
                "auc_test": roc_auc_score(yte, pte),
                "ap_train": average_precision_score(ytr, ptr),
                "ap_test": average_precision_score(yte, pte),
                "base_rate_test": float(np.mean(yte)),
                "n_train": len(tr),
                "n_test": len(te),
                "n_pos_train": n_tr,
                "n_pos_test": n_te,
                "include_gust": INCLUDE_GUST,
            }
        )
    print(f"\n  {'model':8s} {'AUC train':>10s} {'AUC test':>10s} {'AP test':>9s}")
    for r in perf:
        print(f"  {r['model']:8s} {r['auc_train']:10.3f} {r['auc_test']:10.3f} {r['ap_test']:9.3f}")
    print(f"  base rate in test: {perf[0]['base_rate_test']:.3f}")
    perm = permutation_importance(
        rf, Xte, yte, n_repeats=20, random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
    )
    marg = marginal_rho(d, feats, target)
    imp = pd.DataFrame(
        {
            "class": label,
            "target": target,
            "feature": feats,
            "perm_importance": perm.importances_mean,
            "perm_std": perm.importances_std,
            "logit_coef": logit.coef_[0],
            "marginal_rho": [marg[f] for f in feats],
        }
    ).sort_values("perm_importance", ascending=False)
    print(f"\n  {'feature':20s} {'perm imp':>9s} {'logitβ':>8s} {'marg ρ':>8s}  agree")
    for _, r in imp.head(8).iterrows():
        agree = "yes" if np.sign(r["logit_coef"]) == np.sign(r["marginal_rho"]) else "NO"
        print(
            f"  {r['feature']:20s} {r['perm_importance']:9.4f} {r['logit_coef']:8.2f} {r['marginal_rho']:8.3f}  {agree}"
        )
    disagree = imp[np.sign(imp["logit_coef"]) != np.sign(imp["marginal_rho"])]
    top = imp.head(4)
    top_dis = top[np.sign(top["logit_coef"]) != np.sign(top["marginal_rho"])]
    if not top_dis.empty:
        print(f"\n  NOTE: sign disagreement among the top drivers: {', '.join(top_dis['feature'])}")
        print("  Prefer the marginal sign; the coefficient is affected by")
        print("  collinearity with the other predictors.")
    scores = te[["date", "year", target]].copy()
    scores["p_rf"] = rf.predict_proba(Xte)[:, 1]
    scores["class"] = label
    scores["target"] = target
    scores = scores.rename(columns={target: "observed"})
    return (pd.DataFrame(perf), imp, scores)

def regime_reproduction(scores):
    print("\n" + "=" * 70)
    print("REGIME REPRODUCTION (test period, never trained on)")
    for label in scores["class"].unique():
        s = scores[scores["class"] == label]
        y = s.groupby("year").agg(observed=("observed", "mean"), predicted=("p_rf", "mean")).reset_index()
        y["ratio"] = y["predicted"] / y["observed"].replace(0, np.nan)
        print(f"\n{label}")
        print(f"  {'year':6s} {'observed':>10s} {'predicted':>10s} {'ratio':>7s}")
        for _, r in y.iterrows():
            print(f"  {int(r['year']):<6d} {r['observed']:10.4f} {r['predicted']:10.4f} {r['ratio']:7.1f}x")
        if len(y) >= 3:
            corr = np.corrcoef(y["observed"], y["predicted"])[0, 1]
            print(f"  correlation across test years: {corr:+.2f}")
            if corr < 0:
                print("  -> predicted risk moves OPPOSITE to observed.")
                print("     Local meteorology does not explain the decline.")

def plot(importance, scores, axis_deg):
    classes = list(importance["class"].unique())
    n = len(classes)
    fig, axes = plt.subplots(2, n, figsize=(5.2 * n, 8.5))
    if n == 1:
        axes = axes.reshape(2, 1)
    for j, label in enumerate(classes):
        imp = importance[importance["class"] == label].head(8)
        imp = imp.sort_values("perm_importance")
        cols = ["#2c7fb8" if r > 0 else "#c0392b" for r in imp["marginal_rho"].fillna(0)]
        axes[0, j].barh(imp["feature"], imp["perm_importance"], xerr=imp["perm_std"], color=cols)
        axes[0, j].set_title(f"{label}\n(blue = promotes, red = suppresses; marginal sign)", fontsize=9)
        axes[0, j].set_xlabel("Permutation importance (AUC drop)")
        axes[0, j].tick_params(labelsize=7.5)
        s = scores[scores["class"] == label]
        y = s.groupby("year").agg(observed=("observed", "mean"), predicted=("p_rf", "mean")).reset_index()
        ax = axes[1, j]
        ax.plot(y["year"], y["observed"], "o-", color="k", lw=2, label="Observed")
        ax.plot(y["year"], y["predicted"], "s--", color="#238b45", lw=2, label="Predicted risk")
        ax.set_ylim(0, max(y["predicted"].max(), y["observed"].max()) * 1.25)
        ax.set_xticks(y["year"])
        ax.set_xlabel("Year")
        ax.set_ylabel("Rate / risk")
        ax.set_title("Held-out period (shared axis)", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25, ls=":")
    fig.suptitle(
        f"Indicator v2: gust {('included' if INCLUDE_GUST else 'removed')}, wind axis {axis_deg:.0f}°",
        fontsize=10,
    )
    fig.tight_layout()
    fig.savefig(OUT / "INDICATOR_summary.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def main():
    print("NOTEBOOK 06 v2 - RECEPTOR METEOROLOGICAL INDICATOR")
    print("\nScope: receptor-side diagnostic, not an emission model.")
    print("The ERA5 domain stops at 32N and excludes the source regions.")
    df = load()
    axis_deg, provenance = resolve_axis()
    print(f"\nWind projection axis: {axis_deg:.0f} deg")
    print(f"  provenance: {provenance}")
    print(f"Gust term: {('INCLUDED' if INCLUDE_GUST else 'REMOVED (collinear)')}")
    df, feats = engineer(df, axis_deg)
    print(f"\nFeatures ({len(feats)}): {', '.join(feats)}")
    check_vif(df, feats)
    perf_f, imp_f, sc_f = ([], [], [])
    for target, label in TARGETS:
        if target not in df.columns:
            continue
        p, i, s = fit_one(df, feats, target, label)
        if p is None:
            continue
        perf_f.append(p)
        imp_f.append(i)
        sc_f.append(s)
    if not perf_f:
        print("\nNothing fitted.")
        return
    performance = pd.concat(perf_f, ignore_index=True)
    importance = pd.concat(imp_f, ignore_index=True)
    scores = pd.concat(sc_f, ignore_index=True)
    performance.to_csv(OUT / "INDICATOR_performance.csv", index=False)
    importance.to_csv(OUT / "INDICATOR_importance.csv", index=False)
    scores.to_csv(OUT / "INDICATOR_daily_scores.csv", index=False)
    regime_reproduction(scores)
    plot(importance, scores, axis_deg)
    spec = [
        "NOTEBOOK 06 v2 SPECIFICATION",
        f"  wind projection axis : {axis_deg:.0f} deg",
        f"  axis provenance      : {provenance}",
        f"  gust_max included    : {INCLUDE_GUST}",
        f"  features ({len(feats)}) : {', '.join(feats)}",
        f"  train / test         : {TRAIN_YEARS} / {TEST_YEARS}",
        f"  rows after clipping  : {len(df):,} (<= {MAX_YEAR})",
        "",
        "Held-out AUC (random forest):",
    ]
    for _, r in performance[performance["model"] == "rf"].iterrows():
        spec.append(f"  {r['class']:22s} {r['auc_test']:.3f}  (n_pos_test = {r['n_pos_test']})")
    (OUT / "INDICATOR_spec.txt").write_text("\n".join(spec) + "\n")
    print("\n" + "=" * 70)
    print("NEXT")
    print("  1. Check that wind_speed_max now carries a POSITIVE marginal")
    print("     association for every class. If not, stop and inspect.")
    print("  2. Compare the held-out AUC values with the v1 numbers in")
    print("     Table 2 and update the manuscript, including the abstract.")
    print("  3. Re-run Notebook 09 so the figures match.")
    print(f"\n  Specification recorded in {OUT / 'INDICATOR_spec.txt'}")

if __name__ == "__main__":
    main()

## 2. Analysis code
Writes the modules used in Section 3.

In [ ]:
%%writefile /content/rev_pkg/paths.py
import os
from pathlib import Path

PROJECT = Path(
    os.environ.get("RIYADH_PROJECT", "/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution")
)
CLASS_CSV = Path(os.environ.get("CLASS_CSV", PROJECT / "04_Analysis" / "CLASS_daily_events.csv"))
METAR_HOURLY = Path(
    os.environ.get(
        "METAR_HOURLY", PROJECT / "03_Observations" / "METAR" / "processed" / "METAR_hourly_riyadh.csv"
    )
)
HOMOG_CSV = Path(
    os.environ.get(
        "HOMOG_CSV", PROJECT / "03_Observations" / "METAR" / "processed" / "HOMOGENEITY_annual.csv"
    )
)
REV = Path(os.environ.get("REV_DIR", PROJECT / "07_Revision"))
OUT = REV / "outputs"
FIGS = REV / "figures"
NPZ = REV / "REV_traj_cellcounts.npz"
METAR_COMPACT = REV / "REV_metar_compact.csv.gz"
UPWIND_CSV = REV / "REV_upwind_monthly.csv"
NE_DIR = REV / "naturalearth"
for p in (REV, OUT, FIGS, NE_DIR):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/rev_pkg/figstyle.py
import matplotlib as mpl

mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Times New Roman", "Liberation Serif"],
        "font.size": 9,
        "axes.titlesize": 9.5,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": ":",
        "legend.frameon": False,
    }
)
C_SUS = "#2c7fb8"
C_LOC = "#e67e22"
C_BOTH = "#8e6bb3"
C_NEI = "0.70"
C_ALL = "0.25"
SEG = [("Baseline", 2016, 2022), ("Minimum", 2023, 2024), ("Recovery", 2025, 2025)]
SEG_COL = {"Baseline": "0.55", "Minimum": "#c0392b", "Recovery": "#e67e22"}
LAB_SUS = "“Suspension-dominated” (DU/SA)"
LAB_LOC = "Locally raised (BLDU/BLSA)"

def shade_min(ax):
    ax.axvspan(2022.5, 2024.5, color="#c0392b", alpha=0.07, zorder=0, lw=0)

In [ ]:
%%writefile /content/rev_pkg/rev_maps.py
import json
import urllib.request
from pathlib import Path
import numpy as np

RECEPTOR = (24.9578, 46.6989)
ERA5_PT = (24.75, 46.75)
EXCL_KM = 300.0
REGIONS = {
    "Mesopotamia / Fertile Crescent": (30.0, 37.0, 38.0, 49.0),
    "Syrian Desert": (31.0, 35.5, 36.0, 41.0),
    "An Nafud": (27.0, 30.5, 38.0, 43.0),
    "Ad-Dahna corridor": (21.0, 28.0, 44.0, 48.5),
    "Rub al Khali": (16.0, 23.0, 45.0, 56.0),
    "Eastern Arabia / Gulf": (24.0, 30.0, 47.0, 53.0),
    "Iranian plateau / Sistan": (28.0, 35.0, 54.0, 63.0),
    "Northeast Africa": (15.0, 32.0, 25.0, 37.0),
}
NE_URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/{}.geojson"
_CACHE = {}

def _lines(name, ne_dir):
    if name in _CACHE:
        return _CACHE[name]
    p = Path(ne_dir) / f"{name}.geojson"
    if not p.exists():
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(NE_URL.format(name), p)
        except Exception as e:
            print(f"  (coastlines unavailable: {e})")
            _CACHE[name] = []
            return []
    gj = json.loads(p.read_text())
    out = []
    for f in gj["features"]:
        g = f["geometry"]
        parts = [g["coordinates"]] if g["type"] == "LineString" else g["coordinates"]
        for part in parts:
            a = np.asarray(part)
            if ((a[:, 0] > 15) & (a[:, 0] < 70) & (a[:, 1] > 5) & (a[:, 1] < 50)).any():
                out.append(a)
    _CACHE[name] = out
    return out

def basemap(ax, ne_dir, lon=(25, 65), lat=(10, 45), labels=True):
    for a in _lines("ne_50m_coastline", ne_dir):
        ax.plot(a[:, 0], a[:, 1], color="0.25", lw=0.55, zorder=4)
    for a in _lines("ne_50m_admin_0_boundary_lines_land", ne_dir):
        ax.plot(a[:, 0], a[:, 1], color="0.55", lw=0.35, zorder=4)
    ax.set_xlim(*lon)
    ax.set_ylim(*lat)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    if labels:
        ax.set_xlabel("Longitude (°E)")
        ax.set_ylabel("Latitude (°N)")

def excl_circle(ax, color="k", lw=1.1):
    t = np.linspace(0, 2 * np.pi, 200)
    r_lat = EXCL_KM / 111.2
    r_lon = r_lat / np.cos(np.radians(RECEPTOR[0]))
    ax.plot(
        RECEPTOR[1] + r_lon * np.cos(t), RECEPTOR[0] + r_lat * np.sin(t), ls=":", color=color, lw=lw, zorder=6
    )

def receptor(ax, ms=11):
    ax.plot(RECEPTOR[1], RECEPTOR[0], marker="*", ms=ms, color="white", mec="k", mew=0.8, zorder=7, ls="none")

def region_boxes(ax, color="k", lw=0.7, numbers=False):
    from matplotlib.patches import Rectangle

    for i, (name, (la0, la1, lo0, lo1)) in enumerate(REGIONS.items(), 1):
        ax.add_patch(
            Rectangle((lo0, la0), lo1 - lo0, la1 - la0, fill=False, edgecolor=color, lw=lw, ls="--", zorder=5)
        )
        if numbers:
            ax.text(lo0 + 0.3, la1 - 0.3, str(i), fontsize=6.5, va="top", ha="left", zorder=6)

In [ ]:
%%writefile /content/rev_pkg/rev_export.py
import numpy as np
import pandas as pd
import paths as P

TRAJ_DIR = P.PROJECT / "05_HYSPLIT" / "trajectories"
PARQ = P.PROJECT / "06_SourceAttribution" / "PSCF_endpoints.parquet"
R_LAT, R_LON = (24.9578, 46.6989)
LAT_EDGES = np.arange(10.0, 46.0, 1.0)
LON_EDGES = np.arange(25.0, 66.0, 1.0)

def parse_tdump(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            p = line.split()
            if len(p) < 12:
                continue
            try:
                v = [float(x) for x in p]
            except ValueError:
                continue
            rows.append((v[8], v[9], v[10], v[11]))
    return np.array(rows, dtype=np.float32) if rows else None

def export_traj(force=False):
    out = P.NPZ
    if out.exists() and (not force):
        print(
            f"{out.name} already exists ({out.stat().st_size / 1000000.0:.1f} MB); use force=True to rebuild."
        )
        return out
    if PARQ.exists():
        ep = pd.read_parquet(PARQ, columns=["tag", "age", "lat", "lon", "height"])
        print(f"Loaded cached endpoints: {len(ep):,}")
    else:
        files = sorted(TRAJ_DIR.glob("tdump_*"))
        print(f"Parsing {len(files):,} trajectory files (no HYSPLIT rerun) ...")
        frames = []
        for i, f in enumerate(files, 1):
            a = parse_tdump(f)
            if a is not None:
                frames.append(
                    pd.DataFrame(
                        {
                            "tag": f.name.replace("tdump_", ""),
                            "age": a[:, 0],
                            "lat": a[:, 1],
                            "lon": a[:, 2],
                            "height": a[:, 3],
                        }
                    )
                )
            if i % 2000 == 0:
                print(f"  {i:,}/{len(files):,}")
        ep = pd.concat(frames, ignore_index=True)
    tags = np.array(sorted(ep["tag"].unique()))
    ep["ti"] = pd.Series(np.arange(len(tags)), index=tags).loc[ep["tag"].values].values
    lat = ep["lat"].values.astype(np.float64)
    lon = ep["lon"].values.astype(np.float64)
    la0, lo0 = (np.radians(R_LAT), np.radians(R_LON))
    a = (
        np.sin((np.radians(lat) - la0) / 2) ** 2
        + np.cos(la0) * np.cos(np.radians(lat)) * np.sin((np.radians(lon) - lo0) / 2) ** 2
    )
    dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    age_ok = np.abs(ep["age"].values) >= 3
    dist_ok = dist >= 300.0
    i = np.searchsorted(LAT_EDGES, lat, side="right") - 1
    j = np.searchsorted(LON_EDGES, lon, side="right") - 1
    i[lat == LAT_EDGES[-1]] = len(LAT_EDGES) - 2
    j[lon == LON_EDGES[-1]] = len(LON_EDGES) - 2
    in_grid = (i >= 0) & (i < len(LAT_EDGES) - 1) & (j >= 0) & (j < len(LON_EDGES) - 1)
    sel = age_ok & dist_ok & in_grid
    cell = i * (len(LON_EDGES) - 1) + j
    trip = (
        pd.DataFrame({"ti": ep["ti"].values[sel], "cell": cell[sel]})
        .groupby(["ti", "cell"])
        .size()
        .reset_index(name="n")
    )
    per = (
        pd.DataFrame(
            {"ti": ep["ti"].values, "all": 1, "age_ok": age_ok, "age_dist_ok": age_ok & dist_ok, "kept": sel}
        )
        .groupby("ti")[["all", "age_ok", "age_dist_ok", "kept"]]
        .sum()
    )
    per = per.reindex(np.arange(len(tags)), fill_value=0)
    np.savez_compressed(
        out,
        tags=tags.astype("U11"),
        ti=trip["ti"].values.astype(np.int32),
        cell=trip["cell"].values.astype(np.int16),
        n=trip["n"].values.astype(np.int16),
        n_all=per["all"].values.astype(np.int16),
        n_age_ok=per["age_ok"].values.astype(np.int16),
        n_age_dist_ok=per["age_dist_ok"].values.astype(np.int16),
        n_kept=per["kept"].values.astype(np.int16),
        lat_edges=LAT_EDGES,
        lon_edges=LON_EDGES,
    )
    print(f"Trajectories          : {len(tags):,}")
    print(f"Endpoints (all)       : {len(ep):,}")
    print(f"After age + 300 km    : {int((age_ok & dist_ok).sum()):,}")
    print(f"...and inside the grid: {int(sel.sum()):,}")
    print(f"Saved -> {out}  ({out.stat().st_size / 1000000.0:.1f} MB)")
    return out

In [ ]:
%%writefile /content/rev_pkg/rev_stats.py
from math import comb
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
import paths as P

HOM_FALLBACK = pd.DataFrame(
    {
        "year": range(2016, 2026),
        "dust_rate": [0.0465, 0.0704, 0.0843, 0.0613, 0.0511, 0.0436, 0.091, 0.0084, 0.0096, 0.0282],
        "nondust_wx_rate": [0.0197, 0.0168, 0.0235, 0.0251, 0.0147, 0.0105, 0.0192, 0.0333, 0.0132, 0.0215],
        "vis_low_hours": [383, 519, 634, 440, 359, 346, 718, 178, 57, 262],
        "vis_storm_hours": [50, 53, 184, 56, 45, 57, 84, 19, 6, 46],
        "frac_lowvis_coded_dust": [
            0.7467,
            0.8748,
            0.8596,
            0.7909,
            0.8189,
            0.8179,
            0.8231,
            0.236,
            0.8246,
            0.6603,
        ],
    }
)

def load_class():
    d = pd.read_csv(P.CLASS_CSV, parse_dates=["date"])
    d = d[d.date.dt.year <= 2025].copy()
    d["year"], d["month"] = (d.date.dt.year, d.date.dt.month)
    d["L"], d["S"], d["D"] = (d.local_day == 1, d.transport_day == 1, d.dust_day == 1)
    d["both"], d["Lonly"], d["Sonly"] = (d.L & d.S, d.L & ~d.S, d.S & ~d.L)
    d["neither"] = d.D & ~d.L & ~d.S
    return d

def load_hom():
    if P.HOMOG_CSV.exists():
        h = pd.read_csv(P.HOMOG_CSV)
        return h[h.year.between(2016, 2025)].reset_index(drop=True)
    return HOM_FALLBACK.copy()

def class_counts(d):
    out = {}
    for per, (a, b) in {
        "2016-2025": (2016, 2025),
        "2016-2022": (2016, 2022),
        "2023-2025": (2023, 2025),
    }.items():
        s = d[d.year.between(a, b)]
        out[per] = {k: int(s[k].sum()) for k in ["D", "L", "S", "both", "Lonly", "Sonly", "neither"]}
    nl = d[d.D & ~d.L]
    out["dust_days_not_local"] = int(len(nl))
    out["of_which_1_2_local_hours"] = int((nl.local_hours >= 1).sum())
    return out

def table1(d, B=20000, seed=42):
    h = load_hom().rename(
        columns={"nondust_wx_rate": "other_rate", "vis_low_hours": "vis5", "vis_storm_hours": "vis1"}
    )
    cnt = d.groupby("year").agg(days=("D", "sum"), loc=("L", "sum"), sus=("S", "sum")).reset_index()
    t = h[["year", "dust_rate", "other_rate", "vis5", "vis1"]].merge(cnt, on="year")
    t.to_csv(P.OUT / "table1_annual.csv", index=False)
    rng = np.random.default_rng(seed)
    base, mn, rc = (t[t.year <= 2022], t[t.year.isin([2023, 2024])], t[t.year == 2025])
    rows = []
    for c in ["dust_rate", "other_rate", "vis5", "vis1", "days", "loc", "sus"]:
        bv = base[c].values.astype(float)
        boots = rng.choice(bv, (B, len(bv)), replace=True).mean(1)
        for name, sub in [("Min", mn), ("Rec", rc)]:
            v = sub[c].mean()
            ci = np.percentile(100 * (v - boots) / boots, [2.5, 97.5])
            rows.append(
                dict(
                    metric=c,
                    regime=name,
                    base_mean=bv.mean(),
                    base_min=bv.min(),
                    base_max=bv.max(),
                    value=v,
                    pct=100 * (v - bv.mean()) / bv.mean(),
                    ci_lo=ci[0],
                    ci_hi=ci[1],
                )
            )
    ch = pd.DataFrame(rows)
    ch.to_csv(P.OUT / "table1_changes.csv", index=False)
    ranks = {
        c: t.sort_values(c).year.tolist()[:3] for c in ["dust_rate", "vis5", "vis1", "days", "loc", "sus"]
    }
    return (t, ch, ranks, 1 / comb(10, 3))

def prevalence(d):
    rows = []
    for y, g in d.groupby("year"):
        r = {"year": y, "n_days": len(g)}
        for c in ["D", "L", "S", "both", "Lonly", "Sonly"]:
            k = int(g[c].sum())
            lo, hi = proportion_confint(k, len(g), method="wilson")
            r.update({c: k, c + "_pct": 100 * k / len(g), c + "_lo": 100 * lo, c + "_hi": 100 * hi})
        rows.append(r)
    p = pd.DataFrame(rows)
    p.to_csv(P.OUT / "prevalence_by_year.csv", index=False)
    return p

def hourly_checks():
    src = P.METAR_COMPACT if P.METAR_COMPACT.exists() else P.METAR_HOURLY
    if not src.exists():
        return None
    h = pd.read_csv(src, low_memory=False)
    tok = h["wxcodes"].fillna("").astype(str).str.split().apply(set)
    dust = {
        "DU",
        "BLDU",
        "DS",
        "SA",
        "BLSA",
        "SS",
        "PO",
        "VCBLDU",
        "VCBLSA",
        "VCDS",
        "VCSS",
        "+DS",
        "+SS",
        "-DS",
        "-SS",
    }
    is_dust = tok.apply(lambda t: bool(dust & t))
    local = tok.apply(lambda t: bool({"BLDU", "BLSA"} & t))
    susp = tok.apply(lambda t: bool({"DU", "SA"} & t) and (not bool({"BLDU", "BLSA"} & t)))
    storm = tok.apply(lambda t: bool({"DS", "SS"} & t) and (not bool({"BLDU", "BLSA", "DU", "SA"} & t)))
    po = tok.apply(lambda t: "PO" in t and (not bool({"BLDU", "BLSA", "DU", "SA", "DS", "SS"} & t)))
    other = is_dust & ~local & ~susp & ~storm & ~po
    res = {
        "dust_obs": int(is_dust.sum()),
        "local": int(local.sum()),
        "suspension": int(susp.sum()),
        "DS_SS_only": int(storm.sum()),
        "PO_only": int(po.sum()),
        "other_dust_codes": int(other.sum()),
        "vis_km_max": float(h["vis_km"].max()),
        "n_vis_above_9.999": int((h["vis_km"] > 9.9995).sum()),
        "vsby_max_miles": float(pd.to_numeric(h["vsby"], errors="coerce").max()),
    }
    pd.Series(res).to_csv(P.OUT / "hourly_checks.csv")
    return res

if __name__ == "__main__":
    d = load_class()
    print(class_counts(d))
    t, ch, ranks, p = table1(d)
    print(ch.round(2).to_string(index=False))
    print(ranks, p)
    print(prevalence(d).round(1).to_string(index=False))
    print(hourly_checks())

In [ ]:
%%writefile /content/rev_pkg/rev_calib.py
import numpy as np, pandas as pd, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import paths as P

IN = P.CLASS_CSV
BASE = [
    "wind_speed_max",
    "swvl1_mean",
    "precip_sum",
    "precip_7d",
    "precip_30d",
    "vpd_max",
    "blh_max",
    "t2m_max",
    "ssrd_mean",
]
df = pd.read_csv(IN, low_memory=False)
df["date"] = pd.to_datetime(df.date)
df["year"] = df.date.dt.year
df["month"] = df.date.dt.month
df["doy"] = df.date.dt.dayofyear
df = df[df.year <= 2025].copy()
th = np.radians(df.wind_dir_vec)
df["wind_dir_sin"] = np.sin(th)
df["wind_dir_cos"] = np.cos(th)
df["wind_projection"] = df.wind_speed_max * np.cos(th - np.radians(320.0))
df["doy_sin"] = np.sin(2 * np.pi * df.doy / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df.doy / 365.25)
F = BASE + ["wind_dir_sin", "wind_dir_cos", "wind_projection", "doy_sin", "doy_cos"]

def rf():
    return RandomForestClassifier(
        n_estimators=400, min_samples_leaf=5, class_weight="balanced", random_state=0, n_jobs=-1
    )

def lg(x):
    x = np.clip(x, 0.0001, 1 - 0.0001)
    return np.log(x / (1 - x))

rng = np.random.default_rng(7)
out_rows = []
skill = []
store = {}
for target, label in [
    ("local_day", "Locally lifted"),
    ("transport_day", "Suspension-dominated"),
    ("dust_day", "All dust"),
]:
    d = df.dropna(subset=[target] + F).copy()
    tr = d[d.year <= 2022].copy()
    te = d[d.year >= 2023].copy()
    m = rf().fit(tr[F].values, tr[target].values)
    te["s"] = m.predict_proba(te[F].values)[:, 1]
    tr["s"] = np.nan
    for y in range(2016, 2023):
        a = tr[tr.year != y]
        b = tr.year == y
        mm = rf().fit(a[F].values, a[target].values)
        tr.loc[b, "s"] = mm.predict_proba(tr.loc[b, F].values)[:, 1]
    pl = LogisticRegression(C=1000000.0, max_iter=1000).fit(lg(tr.s.values)[:, None], tr[target].values)
    tr["p"] = pl.predict_proba(lg(tr.s.values)[:, None])[:, 1]
    te["p"] = pl.predict_proba(lg(te.s.values)[:, None])[:, 1]
    pbar = tr[target].mean()
    mon = tr.groupby("month")[target].mean()
    for x in (tr, te):
        x["clim"] = pbar
        x["sclim"] = x.month.map(mon)
    yrs = np.arange(2016, 2023)
    Bc = 500
    cal_te = []
    cal_tr = []
    for k in range(Bc):
        ys = rng.choice(yrs, len(yrs), replace=True)
        bb = pd.concat([tr[tr.year == y] for y in ys])
        if bb[target].sum() < 5:
            continue
        plb = LogisticRegression(C=1000000.0, max_iter=1000).fit(lg(bb.s.values)[:, None], bb[target].values)
        cal_te.append(plb.predict_proba(lg(te.s.values)[:, None])[:, 1])
        cal_tr.append(plb.predict_proba(lg(tr.s.values)[:, None])[:, 1])
    cal_te = np.array(cal_te)
    cal_tr = np.array(cal_tr)
    allx = pd.concat([tr, te])
    allcal = np.concatenate([cal_tr, cal_te], axis=1)
    g = tr.groupby("year").agg(O=(target, "sum"), E=("p", "sum"), V=("p", lambda p: (p * (1 - p)).sum()))
    phi = max(1.0, ((g.O - g.E) ** 2).sum() / g.V.sum())
    for y in range(2016, 2026):
        sel = (allx.year == y).values
        x = allx[sel]
        O = x[target].sum()
        E = x.p.sum()
        Ec = x.clim.sum()
        Es = x.sclim.sum()
        Eb = allcal[:, sel].sum(1)
        Vb = (allcal[:, sel] * (1 - allcal[:, sel])).sum(1)
        sims = Eb + np.sqrt(phi * Vb) * rng.standard_normal(len(Eb))
        sims = np.concatenate(
            [Eb[:, None] + np.sqrt(phi * Vb)[:, None] * rng.standard_normal((len(Eb), 40))]
        ).ravel()
        lo, hi = np.percentile(np.clip(sims, 0, None), [2.5, 97.5])
        out_rows.append(
            dict(
                cls=label,
                year=y,
                period="train (OOF)" if y <= 2022 else "test",
                n=len(x),
                O=int(O),
                E_model=E,
                PI_lo=lo,
                PI_hi=hi,
                E_clim=Ec,
                E_sclim=Es,
                O_over_E=O / E if E > 0 else np.nan,
                met_index=E / Es,
                phi=phi,
            )
        )
    yt = te[target].values
    skill.append(
        dict(
            cls=label,
            AUC_model=roc_auc_score(yt, te.s),
            AUC_sclim=roc_auc_score(yt, te.sclim),
            Brier_model=brier_score_loss(yt, te.p),
            Brier_sclim=brier_score_loss(yt, te.sclim),
            Brier_clim=brier_score_loss(yt, te.clim),
            AUC_oof=roc_auc_score(tr[target], tr.s),
            AUC_sclim_train=roc_auc_score(tr[target], tr.sclim),
            Brier_oof=brier_score_loss(tr[target], tr.p),
            Brier_sclim_train=brier_score_loss(tr[target], tr.sclim),
            raw_ratio_train=tr.s.mean() / tr[target].mean(),
            raw_ratio_test=te.s.mean() / te[target].mean(),
            platt_a=pl.intercept_[0],
            platt_b=pl.coef_[0][0],
        )
    )
    store[target] = dict(
        tr=tr[["date", "year", "month", target, "s", "p", "clim", "sclim"]],
        te=te[["date", "year", "month", target, "s", "p", "clim", "sclim"]],
    )
    tr[["date", "year", "month", target, "s", "p", "sclim"]].assign(split="train_oof").to_csv(
        P.OUT / f"calib_{target}_train.csv", index=False
    )
    te[["date", "year", "month", target, "s", "p", "sclim"]].assign(split="test").to_csv(
        P.OUT / f"calib_{target}_test.csv", index=False
    )
R = pd.DataFrame(out_rows)
S = pd.DataFrame(skill)
R.to_csv(P.OUT / "calib_yearly.csv", index=False)
S.to_csv(P.OUT / "calib_skill.csv", index=False)
pd.set_option("display.width", 220)
print(R.round(2).to_string(index=False))
print(S.round(3).T)

In [ ]:
%%writefile /content/rev_pkg/rev_figs.py
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, roc_auc_score
import paths as P
from figstyle import C_SUS, C_LOC, C_BOTH, C_NEI, C_ALL, SEG, SEG_COL, LAB_SUS, LAB_LOC, shade_min
import rev_maps as M
from rev_stats import load_class, load_hom

OUT = P.FIGS
REGIONS = M.REGIONS
RECEPTOR, ERA5_PT = (M.RECEPTOR, M.ERA5_PT)
REG_COL = dict(
    zip(REGIONS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"])
)

def fig1():
    fig, ax = plt.subplots(figsize=(7.4, 5.6))
    M.basemap(ax, P.NE_DIR, lon=(20, 66), lat=(12, 40))
    ax.grid(True, alpha=0.25, ls=":")
    for name, (la0, la1, lo0, lo1) in REGIONS.items():
        c = REG_COL[name]
        ax.add_patch(
            Rectangle(
                (lo0, la0), lo1 - lo0, la1 - la0, facecolor=c, alpha=0.16, edgecolor=c, lw=1.3, zorder=1
            )
        )
    ax.add_patch(Rectangle((35, 18), 20, 14, fill=False, edgecolor="k", lw=1.4, ls="--", zorder=6))
    M.excl_circle(ax, color="tab:blue", lw=1.2)
    ax.plot(
        RECEPTOR[1], RECEPTOR[0], marker="*", ms=15, color="tab:blue", mec="k", mew=0.7, zorder=7, ls="none"
    )
    ax.plot(ERA5_PT[1], ERA5_PT[0], marker="x", ms=6, color="k", mew=1.2, zorder=8, ls="none")
    handles = [Patch(facecolor=REG_COL[n], edgecolor=REG_COL[n], alpha=0.5, label=n) for n in REGIONS]
    handles += [
        Line2D(
            [], [], marker="*", ms=11, color="tab:blue", mec="k", ls="none", label="Receptor, Riyadh OERK"
        ),
        Line2D([], [], color="tab:blue", ls=":", lw=1.2, label="300 km endpoint exclusion radius"),
        Line2D([], [], marker="x", color="k", ls="none", label="ERA5 receptor grid point (24.75°N, 46.75°E)"),
        Line2D([], [], color="k", ls="--", lw=1.4, label="ERA5 hourly domain (18–32°N, 35–55°E)"),
    ]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=7.5, borderaxespad=0)
    fig.savefig(OUT / "Fig1_study_region.png")
    plt.close(fig)

def fig2(d):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.9), gridspec_kw={"width_ratios": [1.05, 1.15, 0.9]})
    cats = [
        ("Sonly", C_SUS, "Suspension-only"),
        ("both", C_BOTH, "Overlap (both classes)"),
        ("Lonly", C_LOC, "Local-only"),
        ("neither", C_NEI, "Neither class"),
    ]
    m = d.groupby("month")[[c for c, _, _ in cats]].sum().reindex(range(1, 13), fill_value=0)
    bottom = np.zeros(12)
    for c, col, lab in cats:
        axes[0].bar(m.index, m[c], bottom=bottom, color=col, width=0.8, label=lab)
        bottom += m[c].values
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(list("JFMAMJJASOND"))
    axes[0].set_xlabel("Month")
    axes[0].set_ylabel("Dust days, 2016–2025")
    axes[0].set_title("(a) Seasonal cycle")
    y = d.groupby("year")[[c for c, _, _ in cats]].sum()
    bottom = np.zeros(len(y))
    for c, col, lab in cats:
        axes[1].bar(y.index, y[c], bottom=bottom, color=col, width=0.75)
        bottom += y[c].values
    axes[1].set_xticks(y.index)
    axes[1].set_xticklabels(y.index, rotation=45)
    axes[1].set_ylabel("Dust days per year")
    axes[1].set_title("(b) Annual counts")
    shade_min(axes[1])
    a = d.groupby("year")[["S", "L"]].sum()
    share = a.S / (a.S + a.L)
    axes[2].plot(a.index, share, "o-", color="#6a3d9a")
    axes[2].axhline(0.5, color="0.6", ls=":", lw=1)
    axes[2].set_ylim(0, 1)
    axes[2].set_xticks(a.index[::2])
    axes[2].set_xlabel("Year")
    axes[2].set_ylabel("S / (S + L)")
    axes[2].set_title("(c) “Suspension-dominated” share of class days")
    shade_min(axes[2])
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout()
    fig.savefig(OUT / "Fig2_event_climatology.png")
    plt.close(fig)

def fig3():
    h = load_hom()
    fig, axes = plt.subplots(3, 1, figsize=(6.8, 7.6), sharex=True)
    axes[0].plot(h.year, h.dust_rate, "o-", color="#c0392b", label="Dust codes")
    axes[0].plot(
        h.year, h.nondust_wx_rate, "s--", color="#2c7fb8", label="Non-dust weather codes (rain, TS, haze)"
    )
    axes[0].set_ylabel("Fraction of\nhourly reports")
    axes[0].set_title("(a) Dust and non-dust weather coding")
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[1].plot(h.year, h.vis_low_hours, "o-", color="#2ca25f", label="V < 5 km")
    axes[1].plot(h.year, h.vis_storm_hours, "s--", color="#006d2c", label="V < 1 km")
    axes[1].set_ylabel("Hours per year")
    axes[1].set_title("(b) Low-visibility hours (instrument record)")
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[2].plot(h.year, h.frac_lowvis_coded_dust, "o-", color="#e67e22")
    axes[2].set_ylim(0, 1)
    axes[2].set_ylabel("Fraction")
    axes[2].set_xlabel("Year")
    axes[2].set_title("(c) Share of V < 5 km hours carrying a dust code")
    for ax in axes:
        shade_min(ax)
    axes[2].set_xticks(h.year)
    fig.tight_layout()
    fig.savefig(OUT / "Fig3_record_homogeneity.png")
    plt.close(fig)

def fig4(d):
    d = d.copy()
    d["is_shamal"] = (d.wind_dir_vec >= 290) & (d.wind_dir_vec <= 350)
    d["shamal_wind"] = np.where(d.is_shamal, d.wind_speed_max, np.nan)
    a = d.groupby("year").agg(
        swvl1=("swvl1_mean", "mean"),
        p30=("precip_30d", "mean"),
        shw=("shamal_wind", "mean"),
        shf=("is_shamal", "mean"),
    )
    h = load_hom().set_index("year")
    fig, axes = plt.subplots(4, 1, figsize=(6.8, 9.2), sharex=True)

    def seg_of(y):
        for n, lo, hi in SEG:
            if lo <= y <= hi:
                return n

    cols = [SEG_COL[seg_of(y)] for y in h.index]
    axes[0].bar(h.index, h.dust_rate, color=cols, width=0.75)
    axes[0].set_ylabel("Dust report\nfraction")
    axes[0].set_title("(a) Observed dust activity")
    handles = [Patch(color=SEG_COL[n]) for n, _, _ in SEG]
    labels = ["Baseline 2016–2022", "Minimum 2023–2024", "Recovery 2025"]
    axes[0].legend(handles, labels, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[0].set_ylim(0, 0.1)
    axes[1].plot(h.index, h.vis_low_hours, "o-", color="#2ca25f", label="V < 5 km")
    axes[1].plot(h.index, h.vis_storm_hours, "s--", color="#006d2c", label="V < 1 km")
    axes[1].set_ylabel("Hours per year")
    axes[1].set_title("(b) Low-visibility hours (independent of coding)")
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    (l1,) = axes[2].plot(a.index, a.swvl1, "o-", color="#2c7fb8", label="Soil moisture (0–7 cm)")
    ax2b = axes[2].twinx()
    (l2,) = ax2b.plot(a.index, a.p30, "^--", color="#41b6c4", label="30-day precipitation")
    ax2b.grid(False)
    axes[2].set_ylabel("Soil moisture\n(m$^3$ m$^{-3}$)")
    ax2b.set_ylabel("30-day precip. (mm)")
    axes[2].set_title("(c) Moisture pathway (ERA5, receptor grid point)")
    axes[2].legend([l1, l2], [l1.get_label(), l2.get_label()], loc="upper left", bbox_to_anchor=(1.2, 1.0))
    (l3,) = axes[3].plot(a.index, a.shw, "o-", color="#238b45", label="Shamal wind speed")
    ax3b = axes[3].twinx()
    (l4,) = ax3b.plot(a.index, a.shf, "^:", color="#00441b", label="Shamal day fraction")
    ax3b.grid(False)
    axes[3].set_ylabel("Shamal wind\n(m s$^{-1}$)")
    ax3b.set_ylabel("Shamal day fraction")
    axes[3].set_xlabel("Year")
    axes[3].set_title("(d) Wind pathway (ERA5, receptor grid point)")
    axes[3].legend([l3, l4], [l3.get_label(), l4.get_label()], loc="upper left", bbox_to_anchor=(1.2, 1.0))
    for ax in axes:
        shade_min(ax)
    axes[3].set_xticks(h.index)
    axes[3].set_xticklabels(h.index, rotation=45)
    fig.tight_layout()
    fig.savefig(OUT / "Fig4_regime_structure.png")
    plt.close(fig)
    return a

BASE = [
    "wind_speed_max",
    "swvl1_mean",
    "precip_sum",
    "precip_7d",
    "precip_30d",
    "vpd_max",
    "blh_max",
    "t2m_max",
    "ssrd_mean",
]
F = BASE + ["wind_dir_sin", "wind_dir_cos", "wind_projection", "doy_sin", "doy_cos"]
PRETTY = {
    "wind_speed_max": "Daily max. wind speed",
    "wind_projection": "Wind projection (320°)",
    "vpd_max": "Daily max. VPD",
    "doy_sin": "Day-of-year (sin)",
    "doy_cos": "Day-of-year (cos)",
    "swvl1_mean": "Soil moisture",
    "t2m_max": "Daily max. temperature",
    "precip_30d": "30-day precipitation",
    "precip_7d": "7-day precipitation",
    "precip_sum": "Daily precipitation",
    "ssrd_mean": "Solar radiation",
    "blh_max": "Max. boundary-layer height",
    "wind_dir_sin": "Wind direction (sin)",
    "wind_dir_cos": "Wind direction (cos)",
}

def features(d):
    d = d.copy()
    d["doy"] = d.date.dt.dayofyear
    th = np.radians(d.wind_dir_vec)
    d["wind_dir_sin"], d["wind_dir_cos"] = (np.sin(th), np.cos(th))
    d["wind_projection"] = d.wind_speed_max * np.cos(th - np.radians(320.0))
    d["doy_sin"] = np.sin(2 * np.pi * d.doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * d.doy / 365.25)
    return d

def rf():
    return RandomForestClassifier(
        n_estimators=400, min_samples_leaf=5, class_weight="balanced", random_state=0, n_jobs=-1
    )

def fig5(d):
    d = features(d)
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0), gridspec_kw={"width_ratios": [1, 1.15, 0.95]})
    bins = np.arange(1.5, 13.5, 1.5)
    d["wbin"] = pd.cut(d.wind_speed_max, bins)
    g = (
        d.groupby("wbin", observed=True)
        .agg(L=("local_day", "mean"), S=("transport_day", "mean"), n=("local_day", "size"))
        .reset_index()
    )
    g = g[g.n >= 20]
    c = [iv.mid for iv in g.wbin]
    axes[0].plot(c, g.S, "s-", color=C_SUS, label="“Suspension-dominated”")
    axes[0].plot(c, g.L, "o-", color=C_LOC, label="Locally raised")
    axes[0].set_xlabel("Daily maximum 10 m wind speed (m s$^{-1}$)")
    axes[0].set_ylabel("Fraction of days in class")
    axes[0].set_title("(a) Response to local wind")
    axes[0].legend(loc="upper left")
    tr, te = (d[d.year <= 2022], d[d.year >= 2023])
    m = rf().fit(tr[F].values, tr.local_day.values)
    perm = permutation_importance(
        m, te[F].values, te.local_day.values, n_repeats=20, random_state=0, scoring="roc_auc", n_jobs=-1
    )
    imp = pd.DataFrame({"f": F, "imp": perm.importances_mean, "sd": perm.importances_std})
    imp["rho"] = [stats.spearmanr(d[f], d.local_day)[0] for f in F]
    imp = imp.nlargest(8, "imp").sort_values("imp")
    cols = ["#2c7fb8" if r > 0 else "#c0392b" for r in imp.rho]
    axes[1].barh([PRETTY[f] for f in imp.f], imp.imp, xerr=imp.sd, color=cols, error_kw=dict(lw=0.8))
    axes[1].set_xlabel("Permutation importance (drop in held-out AUC)")
    axes[1].set_title("(b) Drivers of locally raised days")
    axes[1].legend(
        handles=[
            Line2D([], [], color="#2c7fb8", lw=6, label="positive marginal ρ"),
            Line2D([], [], color="#c0392b", lw=6, label="negative marginal ρ"),
        ],
        loc="lower right",
    )
    imp.to_csv(P.OUT / "fig5_importance_local.csv", index=False)
    for tgt, lab, col in [
        ("local_day", "Locally raised", C_LOC),
        ("transport_day", "“Susp.-dominated”", C_SUS),
    ]:
        mm = rf().fit(tr[F].values, tr[tgt].values)
        s = mm.predict_proba(te[F].values)[:, 1]
        fpr, tpr, _ = roc_curve(te[tgt], s)
        axes[2].plot(fpr, tpr, color=col, label=f"{lab}, RF (AUC {roc_auc_score(te[tgt], s):.3f})")
        clim = te.month.map(tr.groupby("month")[tgt].mean())
        fpr, tpr, _ = roc_curve(te[tgt], clim)
        axes[2].plot(
            fpr,
            tpr,
            color=col,
            ls="--",
            lw=1,
            label=f"{lab}, seasonal clim. (AUC {roc_auc_score(te[tgt], clim):.3f})",
        )
    axes[2].plot([0, 1], [0, 1], ls=":", color="0.6", lw=1)
    axes[2].set_xlabel("False positive rate")
    axes[2].set_ylabel("True positive rate")
    axes[2].set_title("(c) Held-out skill, 2023–2025")
    axes[2].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=7, ncol=1)
    axes[2].set_aspect("equal")
    fig.tight_layout()
    fig.savefig(OUT / "Fig5_class_skill.png")
    plt.close(fig)
    return imp

def fig6():
    r = pd.read_csv(P.OUT / "calib_yearly.csv")
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.9), sharey=False)
    for ax, (cls, col, title) in zip(
        axes,
        [
            ("Locally lifted", C_LOC, "(a) Locally raised"),
            ("Suspension-dominated", C_SUS, "(b) “Suspension-dominated”"),
            ("All dust", C_ALL, "(c) All dust days"),
        ],
    ):
        s = r[r.cls == cls].sort_values("year")
        ax.fill_between(
            s.year, s.PI_lo, s.PI_hi, color=col, alpha=0.18, lw=0, label="Calibrated model, 95% PI"
        )
        ax.plot(s.year, s.E_model, "--", color=col, lw=1.6, label="Calibrated model, expected")
        ax.plot(s.year, s.E_sclim, ":", color="0.35", lw=1.2, label="Seasonal climatology")
        ax.plot(s.year, s.O, "o-", color="k", lw=1.6, ms=4.5, label="Observed")
        ax.axvline(2022.5, color="0.5", lw=0.8)
        shade_min(ax)
        ax.set_xticks(range(2016, 2026))
        ax.set_xticklabels(range(2016, 2026), rotation=45)
        ax.set_ylim(0, max(s.PI_hi.max(), s.O.max()) * 1.08)
        ax.set_ylabel("Days per year")
        ax.set_title(title)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout()
    fig.savefig(OUT / "Fig6_calibrated_expectation.png")
    plt.close(fig)

def make_all():
    d = load_class()
    fig1()
    fig2(d)
    fig3()
    fig4(d)
    fig5(d)
    fig6()
    print("Figures 1-6 written to", OUT)

In [ ]:
%%writefile /content/rev_pkg/rev_traj.py
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse

RECEPTOR = (24.9578, 46.6989)
EXCL_KM = 300.0
SEASON = [3, 4, 5, 6]
PRIMARY = (2016, 2022)
REGIONS = {
    "Syrian Desert": (31.0, 35.5, 36.0, 41.0),
    "An Nafud": (27.0, 30.5, 38.0, 43.0),
    "Rub al Khali": (16.0, 23.0, 45.0, 56.0),
    "Mesopotamia / Fertile Crescent": (30.0, 37.0, 38.0, 49.0),
    "Ad-Dahna corridor": (21.0, 28.0, 44.0, 48.5),
    "Northeast Africa": (15.0, 32.0, 25.0, 37.0),
    "Eastern Arabia / Gulf": (24.0, 30.0, 47.0, 53.0),
    "Iranian plateau / Sistan": (28.0, 35.0, 54.0, 63.0),
}
THRESH = 1.5
EXT_FLOOR = 3.912 / 9.994

def load_counts(npz_path):
    z = np.load(npz_path)
    lat_e, lon_e = (z["lat_edges"], z["lon_edges"])
    ncell = (len(lat_e) - 1) * (len(lon_e) - 1)
    tags = z["tags"].astype(str)
    C = sparse.csr_matrix(
        (z["n"].astype(np.float64), (z["ti"], z["cell"].astype(np.int64))), shape=(len(tags), ncell)
    )
    meta = pd.DataFrame(
        {
            "tag": tags,
            "n_all": z["n_all"],
            "n_age_ok": z["n_age_ok"],
            "n_age_dist_ok": z["n_age_dist_ok"],
            "n_kept": z["n_kept"],
        }
    )
    meta["date"] = pd.to_datetime(meta.tag.str[:8], format="%Y%m%d")
    meta["utc_hour"] = meta.tag.str[9:11].astype(int)
    meta["year"] = meta.date.dt.year
    meta["month"] = meta.date.dt.month
    return (C, meta, lat_e, lon_e)

def attach_days(meta, class_csv):
    d = pd.read_csv(class_csv, parse_dates=["date"])
    keep = [
        "date",
        "local_day",
        "transport_day",
        "dust_day",
        "wind_speed_max",
        "extinction_max",
        "vis_min_km",
        "local_hours",
        "transport_hours",
        "dust_hours",
    ]
    d = d[[c for c in keep if c in d.columns]]
    m = meta.merge(d, on="date", how="left")
    m["L"] = (m.local_day == 1).astype(float)
    m["S"] = (m.transport_day == 1).astype(float)
    m["Lonly"] = ((m.L == 1) & (m.S == 0)).astype(float)
    m["Sonly"] = ((m.S == 1) & (m.L == 0)).astype(float)
    m["both"] = ((m.S == 1) & (m.L == 1)).astype(float)
    m["D"] = (m.dust_day == 1).astype(float)
    m.loc[m.local_day.isna(), ["L", "S", "Lonly", "Sonly", "both", "D"]] = np.nan
    return m

def dust_hour_intensity(meta, metar_csv, window_h=3):
    h = pd.read_csv(metar_csv, low_memory=False)
    h["valid"] = pd.to_datetime(h["valid"], utc=True, errors="coerce")
    h = h.dropna(subset=["valid"])
    toks = h["wxcodes"].fillna("").astype(str).str.split()
    dust = {
        "DU",
        "BLDU",
        "DS",
        "SA",
        "BLSA",
        "SS",
        "PO",
        "VCBLDU",
        "VCBLSA",
        "VCDS",
        "VCSS",
        "+DS",
        "+SS",
        "-DS",
        "-SS",
    }
    h["is_dust"] = toks.apply(lambda t: bool(dust.intersection(t)))
    h["ext"] = 3.912 / h["vis_km"].clip(lower=0.05)
    hd = h[h.is_dust].copy()
    hd["local_date"] = (hd.valid + pd.Timedelta(hours=3)).dt.tz_localize(None).dt.normalize()
    day_max = hd.groupby("local_date")["ext"].max()
    arr = (meta.date + pd.to_timedelta(meta.utc_hour, unit="h")).dt.tz_localize("UTC")
    t = hd.valid.values.astype("datetime64[s]").astype(np.int64)
    e = hd.ext.values
    order = np.argsort(t)
    t, e = (t[order], e[order])
    a = arr.values.astype("datetime64[s]").astype(np.int64)
    lo = np.searchsorted(t, a - window_h * 3600, side="left")
    hi = np.searchsorted(t, a + window_h * 3600, side="right")
    out = np.full(len(a), EXT_FLOOR)
    for i in np.where(hi > lo)[0]:
        out[i] = e[lo[i] : hi[i]].max()
    return (out, meta.date.map(day_max).fillna(EXT_FLOOR).values)

def polissar(n):
    occ = n[n > 0]
    base = np.median(occ) if occ.size else np.nan
    w = np.full(n.shape, 0.05)
    w[n > base] = 0.42
    w[n > 1.5 * base] = 0.7
    w[n > 3.0 * base] = 1.0
    w[n == 0] = 0.0
    return (w, base)

def corr(a, b, mask):
    x, y = (a[mask], b[mask])
    ok = np.isfinite(x) & np.isfinite(y)
    return float(np.corrcoef(x[ok], y[ok])[0, 1]) if ok.sum() > 20 else np.nan

def region_masks(lat_e, lon_e):
    lat_m = (lat_e[:-1] + lat_e[1:]) / 2
    lon_m = (lon_e[:-1] + lon_e[1:]) / 2
    LA, LO = np.meshgrid(lat_m, lon_m, indexing="ij")
    masks = {}
    for name, (a0, a1, o0, o1) in REGIONS.items():
        masks[name] = ((LA >= a0) & (LA <= a1) & (LO >= o0) & (LO <= o1)).ravel()
    masks["All eight regions"] = np.any(np.vstack(list(masks.values())), axis=0)
    masks["Whole domain"] = np.ones(LA.size, dtype=bool)
    dlat = np.radians(LA - RECEPTOR[0])
    dlon = np.radians(LO - RECEPTOR[1])
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(RECEPTOR[0])) * np.cos(np.radians(LA)) * np.sin(dlon / 2) ** 2
    )
    dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return (masks, dist.ravel())

In [ ]:
%%writefile /content/rev_pkg/rev_traj_run.py
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import rev_traj as R

warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="invalid value encountered")
CLASSES = ["L", "S", "Lonly", "Sonly"]
PAIRS = [("L", "S"), ("Lonly", "Sonly")]
DESIGNS = ["unmatched", "season", "wind"]

def _intensities(meta, metar):
    meta["dC_daymax"] = (meta.extinction_max - R.EXT_FLOOR).clip(lower=0).fillna(0)
    meta["dC_hours_L"] = meta.local_hours.fillna(0)
    meta["dC_hours_S"] = meta.transport_hours.fillna(0)
    kinds = ["daymax", "hours"]
    if metar is not None and Path(metar).exists():
        arr, dmax = R.dust_hour_intensity(meta, metar, window_h=3)
        meta["dC_arrival"] = np.clip(arr - R.EXT_FLOOR, 0, None)
        meta["dC_dustday"] = np.clip(dmax - R.EXT_FLOOR, 0, None)
        kinds += ["arrival", "dustday"]
    return kinds

def _intensity_vec(meta, c, k):
    if k == "hours":
        return meta["dC_hours_L" if c.startswith("L") else "dC_hours_S"].values
    return meta[f"dC_{k}"].values

def _std_field(C, v, strata_idx, nb_mat):
    num = np.asarray(C.T @ v).ravel()
    means = np.array([v[strata_idx == j].mean() for j in range(nb_mat.shape[0])])
    exp = means @ nb_mat
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(exp > 0, num / exp, np.nan)

def run_all(npz, class_csv, outdir, metar=None, B=1000, seed=20260928, verbose=True):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    C, meta, lat_e, lon_e = R.load_counts(npz)
    meta = R.attach_days(meta, class_csv)
    kinds = _intensities(meta, metar)
    masks, dist = R.region_masks(lat_e, lon_e)
    res = {
        "n_traj_files": int(len(meta)),
        "endpoints_all": int(meta.n_all.sum()),
        "endpoints_after_filters": int(meta.n_age_dist_ok.sum()),
        "endpoints_in_grid": int(meta.n_kept.sum()),
        "traj_without_endpoints_all_years": int((meta.n_age_dist_ok == 0).sum()),
        "intensities": kinds,
    }
    prim = meta.year.between(*R.PRIMARY) & meta.L.notna()
    keep = prim & (meta.n_age_dist_ok > 0)
    des = {"unmatched": keep.values, "season": (keep & meta.month.isin(R.SEASON)).values}
    des["wind"] = des["season"].copy()
    sel = np.where(des["wind"])[0]
    q = pd.qcut(meta.wind_speed_max.values[sel], 5, labels=False, duplicates="drop")
    strata = {d: np.zeros(len(meta), dtype=int) for d in DESIGNS}
    strata["wind"][:] = -1
    strata["wind"][sel] = q
    res["dropped_no_endpoint"] = {
        "all_months_2016_2022": int((prim & (meta.n_age_dist_ok == 0)).sum()),
        "MarJun_2016_2022": int((prim & meta.month.isin(R.SEASON) & (meta.n_age_dist_ok == 0)).sum()),
    }
    res["design_sizes"] = {}
    for d in DESIGNS:
        s = meta[des[d]]
        res["design_sizes"][d] = {
            "traj": int(len(s)),
            "days": int(s.date.nunique()),
            **{f"{c}_traj": int(s[c].sum()) for c in CLASSES},
            **{f"{c}_days": int(s.drop_duplicates("date")[c].sum()) for c in CLASSES},
        }
    per = meta.iloc[sel].copy()
    per["q"] = q
    rows = []
    for b in range(int(q.max()) + 1):
        s = per[per.q == b]
        dd = s.drop_duplicates("date")
        r = {
            "quintile": b + 1,
            "wind_lo": float(s.wind_speed_max.min()),
            "wind_hi": float(s.wind_speed_max.max()),
            "days": int(len(dd)),
            "traj": int(len(s)),
        }
        for c in ["S", "L", "both", "Sonly", "Lonly"]:
            r[f"{c}_days"] = int(dd[c].sum())
            r[f"{c}_traj"] = int(s[c].sum())
        r["p_S"] = r["S_traj"] / r["traj"]
        r["p_L"] = r["L_traj"] / r["traj"]
        r["nonS_traj"] = r["traj"] - r["S_traj"]
        r["nonL_traj"] = r["traj"] - r["L_traj"]
        rows.append(r)
    t5 = pd.DataFrame(rows)
    t5.to_csv(out / "table5_counts_by_quintile.csv", index=False)
    mon = per.drop_duplicates("date").groupby("month")[["S", "L", "both", "Sonly", "Lonly"]].sum().astype(int)
    mon.to_csv(out / "counts_by_month_MarJun.csv")
    allm = meta[des["unmatched"]].drop_duplicates("date").copy()
    allm["season_name"] = allm.month.map(
        {
            12: "DJF",
            1: "DJF",
            2: "DJF",
            3: "MAM",
            4: "MAM",
            5: "MAM",
            6: "JJA",
            7: "JJA",
            8: "JJA",
            9: "SON",
            10: "SON",
            11: "SON",
        }
    )
    allm.groupby("season_name")[["S", "L", "both", "Sonly", "Lonly"]].sum().astype(int).to_csv(
        out / "counts_by_season_2016_2022.csv"
    )
    F, design_cache = ({}, {})
    for d in DESIGNS:
        idx = np.where(des[d])[0]
        Cd = C[idx].tocsr()
        st = strata[d][idx]
        sids = np.unique(st)
        st_idx = np.searchsorted(sids, st)
        nb_mat = np.vstack([np.asarray(Cd[st_idx == j].sum(axis=0)).ravel() for j in range(len(sids))])
        n = nb_mat.sum(axis=0)
        w, base_n = R.polissar(n)
        design_cache[d] = (idx, Cd, st_idx, nb_mat, n, w)
        for c in CLASSES:
            lab = meta[c].values[idx].astype(float)
            f = {"n": n, "w": w, "base_n": base_n, "p": float(lab.mean())}
            f["E_raw"] = _std_field(Cd, lab, st_idx, nb_mat)
            f["E"] = np.nan_to_num(f["E_raw"]) * w
            for k in kinds:
                v = lab * _intensity_vec(meta, c, k)[idx]
                f[f"cwt_{k}_raw"] = _std_field(Cd, v, st_idx, nb_mat)
                f[f"cwt_{k}"] = np.nan_to_num(f[f"cwt_{k}_raw"]) * w
                num = np.asarray(Cd.T @ v).ravel()
                with np.errstate(divide="ignore", invalid="ignore"):
                    f[f"cwt_{k}_abs"] = np.where(n > 0, num / n, 0.0) * w
            if d == "wind":
                num_p = np.zeros_like(n)
                den_p = np.zeros_like(n)
                ne = nc = 0
                for j in range(len(sids)):
                    s = st_idx == j
                    nev, nct = (int(lab[s].sum()), int((1 - lab[s]).sum()))
                    if nev < 15 or nct < 15:
                        continue
                    mb = np.asarray(Cd[s].T @ lab[s]).ravel()
                    num_p += mb * nev
                    den_p += nb_mat[j] * nev
                    ne += nev
                    nc += nct
                pU = ne / max(1, ne + nc)
                used_b = [
                    j
                    for j in range(len(sids))
                    if lab[st_idx == j].sum() >= 15 and (1 - lab[st_idx == j]).sum() >= 15
                ]
                n_used = nb_mat[used_b].sum(axis=0)
                w_used, _ = R.polissar(n_used)
                with np.errstate(divide="ignore", invalid="ignore"):
                    f["Epool_raw"] = np.where(den_p > 0, num_p / den_p / pU, np.nan)
                f["Epool"] = np.nan_to_num(f["Epool_raw"]) * w_used
                f["n_used"] = n_used
                f["w_used"] = w_used
                pb = np.array([lab[st_idx == j].mean() for j in range(len(sids))])
                Ncb = np.array([lab[st_idx == j].sum() for j in range(len(sids))])
                use = np.array(
                    [Ncb[j] >= 15 and (st_idx == j).sum() - Ncb[j] >= 15 for j in range(len(sids))]
                )
                ex_num = ((Ncb * pb * use)[:, None] * nb_mat).sum(0)
                ex_den = ((Ncb * use)[:, None] * nb_mat).sum(0)
                with np.errstate(divide="ignore", invalid="ignore"):
                    f["Epool_nullexp"] = np.where(ex_den > 0, ex_num / ex_den / pU, np.nan)
            F[d, c] = f

    def r_w(a, b, fa, fb):
        return R.corr(a, b, (fa["n"] > 0) & (fb["n"] > 0))

    def r_u(a, b, f):
        return R.corr(a, b, (f["w"] >= 0.7) & (f["n"] > 0))

    corr_rows = []
    for a, b in [("L", "S"), ("Lonly", "Sonly"), ("L", "Lonly"), ("S", "Sonly")]:
        for d in DESIGNS:
            fa, fb = (F[d, a], F[d, b])
            row = {
                "pair": f"{a}-{b}",
                "design": d,
                "PSCF_weighted": r_w(fa["E"], fb["E"], fa, fb),
                "PSCF_unweighted_W>=0.7": r_u(fa["E_raw"], fb["E_raw"], fa),
            }
            if d == "wind":
                row["PSCF_pooled_weighted"] = R.corr(
                    fa["Epool"], fb["Epool"], (fa["n_used"] > 0) & (fb["n_used"] > 0)
                )
            for k in kinds:
                row[f"CWT_{k}_weighted"] = r_w(fa[f"cwt_{k}"], fb[f"cwt_{k}"], fa, fb)
                row[f"CWT_{k}_unweighted_W>=0.7"] = r_u(fa[f"cwt_{k}_raw"], fb[f"cwt_{k}_raw"], fa)
            corr_rows.append(row)
    corr_df = pd.DataFrame(corr_rows)
    corr_df.to_csv(out / "class_field_correlations.csv", index=False)
    fields_null = ["PSCF"] + [f"CWT_{k}" for k in kinds]
    null_r, null_rows, region_rows, sig, contrast_rows = ({}, [], [], {}, [])
    for d in DESIGNS:
        idx, Cd, st_idx, nb_mat, n, w = design_cache[d]
        sub = meta.iloc[idx]
        days, day_of = np.unique(sub.date.values, return_inverse=True)
        day_st = pd.Series(st_idx).groupby(day_of).first().values
        vecs = {}
        for c in CLASSES:
            lab_d = pd.Series(sub[c].values).groupby(day_of).first().values.astype(float)
            vecs["PSCF", c] = lab_d
            for k in kinds:
                inten = pd.Series(_intensity_vec(meta, c, k)[idx]).groupby(day_of).first().values
                vecs[f"CWT_{k}", c] = lab_d * inten
        keys = list(vecs)
        Vd = np.vstack([vecs[kk] for kk in keys])
        CdT = Cd.T.tocsr()
        ncell = Cd.shape[1]
        occ = n > 0
        acc = {kk: np.empty((B, ncell), dtype=np.float32) for kk in keys}
        groups = [np.where(day_st == g)[0] for g in np.unique(day_st)]
        chunk = 50
        for k0 in range(0, B, chunk):
            kk_n = min(chunk, B - k0)
            blocks = []
            for _ in range(kk_n):
                src = np.arange(len(days))
                for g in groups:
                    src[g] = g[rng.permutation(len(g))]
                blocks.append(Vd[:, src][:, day_of])
            Vt = np.concatenate(blocks, axis=0)
            M = (CdT @ Vt.T).T
            means = np.stack([Vt[:, st_idx == j].mean(axis=1) for j in range(nb_mat.shape[0])], axis=1)
            EXP = means @ nb_mat
            with np.errstate(divide="ignore", invalid="ignore"):
                Er = np.where(EXP > 0, M / EXP, np.nan).astype(np.float32)
            Er = Er.reshape(kk_n, len(keys), ncell)
            for i, kk in enumerate(keys):
                acc[kk][k0 : k0 + kk_n] = Er[:, i, :]
        for fld in fields_null:
            for a, b in PAIRS:
                Na, Nb = (acc[fld, a], acc[fld, b])
                rw = np.array(
                    [R.corr(np.nan_to_num(Na[k]) * w, np.nan_to_num(Nb[k]) * w, occ) for k in range(B)]
                )
                ru = np.array([R.corr(Na[k], Nb[k], (w >= 0.7) & occ) for k in range(B)])
                null_r[f"{d}__{fld}__{a}-{b}__weighted"] = rw
                null_r[f"{d}__{fld}__{a}-{b}__unweighted"] = ru
                ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
                fa, fb = (F[d, a], F[d, b])
                ow = r_w(fa[ka], fb[ka], fa, fb)
                ou = r_u(fa[ka + "_raw"], fb[ka + "_raw"], fa)
                null_rows.append(
                    {
                        "design": d,
                        "field": fld,
                        "pair": f"{a}-{b}",
                        "r_obs_weighted": ow,
                        "null_w_mean": np.nanmean(rw),
                        "null_w_lo": np.nanpercentile(rw, 2.5),
                        "null_w_hi": np.nanpercentile(rw, 97.5),
                        "p_w_greater": (np.sum(rw >= ow) + 1) / (B + 1),
                        "r_obs_unweighted": ou,
                        "null_u_mean": np.nanmean(ru),
                        "null_u_lo": np.nanpercentile(ru, 2.5),
                        "null_u_hi": np.nanpercentile(ru, 97.5),
                        "p_u_greater": (np.sum(ru >= ou) + 1) / (B + 1),
                    }
                )
        for fld in fields_null:
            for c in CLASSES:
                f = F[d, c]
                ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
                Eo_raw, Eo = (f[ka + "_raw"], f[ka])
                Nn = acc[fld, c]
                q95 = np.nanpercentile(Nn, 95, axis=0)
                s_cell = (Eo_raw > q95) & occ
                sig[d, fld, c] = s_cell
                Nw = np.nan_to_num(Nn) * w
                null_sig_frac = Nn > q95[None, :]
                for reg, rm in masks.items():
                    o = rm & occ
                    if o.sum() == 0:
                        continue
                    obs_mean = float(np.mean(Eo[o]))
                    null_means = Nw[:, o].mean(axis=1)
                    cross = np.asarray(Cd[:, np.where(rm)[0]].sum(axis=1)).ravel() > 0
                    subc = sub[cross]
                    region_rows.append(
                        {
                            "design": d,
                            "field": fld,
                            "class": c,
                            "region": reg,
                            "cells_occupied": int(o.sum()),
                            "cells_W1": int((o & (w >= 1.0)).sum()),
                            "cells_W>=0.7": int((o & (w >= 0.7)).sum()),
                            "median_n_ij": float(np.median(n[o])),
                            "endpoint_share": float(n[rm].sum() / n.sum()),
                            "excluded_frac": float((dist[rm] < R.EXCL_KM).mean()),
                            "traj_crossing": int(cross.sum()),
                            "class_traj_crossing": int(subc[c].sum()),
                            "class_days_crossing": int(subc.drop_duplicates("date")[c].sum()),
                            "mean": obs_mean,
                            "frac_gt_1p5": float((Eo[o] > R.THRESH).mean()),
                            "null_mean": float(np.mean(null_means)),
                            "null_lo": float(np.percentile(null_means, 2.5)),
                            "null_hi": float(np.percentile(null_means, 97.5)),
                            "p_region": float((np.sum(null_means >= obs_mean) + 1) / (B + 1)),
                            "null_frac_gt_1p5": float(np.mean((Nw[:, o] > R.THRESH).mean(axis=1))),
                            "frac_cells_sig": float(s_cell[o].mean()),
                            "null_frac_cells_sig": float(null_sig_frac[:, o].mean()),
                            "null_frac_cells_sig_hi": float(
                                np.percentile(null_sig_frac[:, o].mean(axis=1), 97.5)
                            ),
                            "p_field_sig": float(
                                (np.sum(null_sig_frac[:, o].mean(axis=1) >= s_cell[o].mean()) + 1) / (B + 1)
                            ),
                            "mean_unweighted_W>=0.42": (
                                float(np.nanmean(Eo_raw[o & (w >= 0.42)]))
                                if (o & (w >= 0.42)).any()
                                else np.nan
                            ),
                        }
                    )
        for fld in fields_null:
            ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
            for a, b in [("S", "L"), ("Sonly", "Lonly")]:
                Na = np.nan_to_num(acc[fld, a]) * w
                Nb = np.nan_to_num(acc[fld, b]) * w
                for reg, rm in masks.items():
                    o = rm & occ
                    if o.sum() == 0:
                        continue
                    obs = float(F[d, a][ka][o].mean() - F[d, b][ka][o].mean())
                    nd = Na[:, o].mean(axis=1) - Nb[:, o].mean(axis=1)
                    contrast_rows.append(
                        {
                            "design": d,
                            "field": fld,
                            "contrast": f"{a} minus {b}",
                            "region": reg,
                            "obs_diff": obs,
                            "null_mean": float(nd.mean()),
                            "null_lo": float(np.percentile(nd, 2.5)),
                            "null_hi": float(np.percentile(nd, 97.5)),
                            "p_susp_greater": float((np.sum(nd >= obs) + 1) / (B + 1)),
                            "p_local_greater": float((np.sum(nd <= obs) + 1) / (B + 1)),
                        }
                    )
        if verbose:
            print(f"  permutation nulls done: {d}")
    null_df = pd.DataFrame(null_rows)
    null_df.to_csv(out / "permutation_null_correlations.csv", index=False)
    reg_df = pd.DataFrame(region_rows)
    reg_df.to_csv(out / "region_summary_with_nulls.csv", index=False)
    np.savez_compressed(out / "null_correlations.npz", **null_r)
    pd.DataFrame(contrast_rows).to_csv(out / "negative_control_contrasts.csv", index=False)
    rep = []
    for c in ["S", "L"]:
        f = F["wind", c]
        for reg, rm in masks.items():
            o = rm & (f["n"] > 0)
            ou = rm & (f["n_used"] > 0)
            if o.sum():
                rep.append(
                    {
                        "class": c,
                        "region": reg,
                        "pooled_mean": float(f["Epool"][ou].mean()),
                        "pooled_frac_gt_1p5": float((f["Epool"][ou] > 1.5).mean()),
                        "pooled_null_expectation_mean": float(np.nanmean(f["Epool_nullexp"][ou])),
                        "pooled_cells": int(ou.sum()),
                        "pooled_endpoint_share": float(f["n_used"][rm].sum() / f["n_used"].sum()),
                        "std_mean": float(f["E"][o].mean()),
                        "std_frac_gt_1p5": float((f["E"][o] > 1.5).mean()),
                        "std_mean_null": float(
                            reg_df[
                                (reg_df.design == "wind")
                                & (reg_df.field == "PSCF")
                                & (reg_df["class"] == c)
                                & (reg_df.region == reg)
                            ]["null_mean"].iloc[0]
                        ),
                        "cells": int(o.sum()),
                        "endpoint_share": float(f["n"][rm].sum() / f["n"].sum()),
                    }
                )
    pd.DataFrame(rep).to_csv(out / "pooled_vs_standardised_regions.csv", index=False)
    res["pooled_null_expectation_domain_mean"] = {
        c: float(np.nanmean(F["wind", c]["Epool_nullexp"][F["wind", c]["n_used"] > 0])) for c in CLASSES
    }
    save = {"lat_edges": lat_e, "lon_edges": lon_e}
    for (d, c), f in F.items():
        for k, v in f.items():
            if isinstance(v, np.ndarray):
                save[f"{d}__{c}__{k}"] = v
    for (d, fld, c), v in sig.items():
        save[f"sig__{d}__{fld}__{c}"] = v
    np.savez_compressed(out / "fields_revision.npz", **save)
    res["base_rates"] = {f"{d}:{c}": F[d, c]["p"] for d, c in F}
    res["base_n_median"] = {d: float(F[d, "L"]["base_n"]) for d in DESIGNS}
    res["wind_quintile_edges"] = t5[["wind_lo", "wind_hi"]].round(3).values.tolist()
    json.dump(res, open(out / "summary.json", "w"), indent=1, default=float)
    if verbose:
        print(json.dumps(res, indent=1, default=float)[:2500])
        print(corr_df.round(3).to_string())
        print(null_df.round(3).to_string())
    return res

In [ ]:
%%writefile /content/rev_pkg/rev_traj_figs.py
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import figstyle
from figstyle import C_SUS, C_LOC, C_BOTH
import rev_maps as M

DES_LAB = {
    "unmatched": "Unmatched\n(all months)",
    "season": "Season-matched\n(Mar–Jun)",
    "wind": "Wind-matched\n(Mar–Jun, quintiles)",
}
PAIR_STYLE = {
    "L-S": dict(
        color=C_BOTH, marker="o", label="Overlapping classes (locally raised vs “suspension-dominated”)"
    ),
    "Lonly-Sonly": dict(
        color="#1b7837", marker="s", label="Mutually exclusive classes (local-only vs suspension-only)"
    ),
}
CLS_LAB = {
    "S": "“Suspension-dominated”",
    "L": "Locally raised",
    "Sonly": "Suspension-only",
    "Lonly": "Local-only",
}

def _load(res_dir):
    res_dir = Path(res_dir)
    z = np.load(res_dir / "fields_revision.npz")
    nul = pd.read_csv(res_dir / "permutation_null_correlations.csv")
    cor = pd.read_csv(res_dir / "class_field_correlations.csv")
    return (z, nul, cor)

def fig7(res_dir, out_png, fields=(("PSCF", "PSCF"), ("CWT_daymax", "CWT"))):
    res_dir = Path(res_dir)
    nul = pd.read_csv(res_dir / "permutation_null_correlations.csv")
    con = pd.read_csv(res_dir / "negative_control_contrasts.csv")
    con = con[con.region == "All eight regions"]
    cols = [("w", "weighted r"), ("u", "unweighted r"), ("c", "contrast")]
    designs = ["unmatched", "season", "wind"]
    fig, axes = plt.subplots(
        len(fields), 3, figsize=(7.4, 2.55 * len(fields) + 0.75), sharex=True, squeeze=False
    )
    k = 0
    for i, (fld, flab) in enumerate(fields):
        for j, (kind, clab) in enumerate(cols):
            ax = axes[i, j]
            for jj, pair in enumerate(["L-S", "Lonly-Sonly"]):
                st = PAIR_STYLE[pair]
                dx = -0.15 if jj == 0 else 0.15
                for x, d in enumerate(designs):
                    if kind == "c":
                        cname = "S minus L" if pair == "L-S" else "Sonly minus Lonly"
                        r = con[(con.design == d) & (con.field == fld) & (con.contrast == cname)]
                        if r.empty:
                            continue
                        r = r.iloc[0]
                        lo, hi, mu, obs = (r.null_lo, r.null_hi, r.null_mean, r.obs_diff)
                    else:
                        r = nul[(nul.design == d) & (nul.field == fld) & (nul.pair == pair)]
                        if r.empty:
                            continue
                        r = r.iloc[0]
                        lo, hi, mu = (r[f"null_{kind}_lo"], r[f"null_{kind}_hi"], r[f"null_{kind}_mean"])
                        obs = r["r_obs_weighted" if kind == "w" else "r_obs_unweighted"]
                    ax.add_patch(
                        plt.Rectangle((x + dx - 0.09, lo), 0.18, hi - lo, color="0.82", lw=0, zorder=1)
                    )
                    ax.plot([x + dx - 0.09, x + dx + 0.09], [mu, mu], color="0.35", lw=1.0, zorder=2)
                    ax.plot(
                        x + dx,
                        obs,
                        marker=st["marker"],
                        color=st["color"],
                        ms=5.5,
                        mec="k",
                        mew=0.5,
                        zorder=3,
                        ls="none",
                    )
            ax.axhline(0, color="0.5", lw=0.6)
            ax.set_xlim(-0.5, 2.5)
            ax.set_ylim((-0.45, 0.45) if kind == "c" else (-0.4, 1.0))
            ax.set_xticks(range(3))
            ax.set_xticklabels(["Unmatched", "Season", "Wind"], fontsize=7.5)
            ax.set_title(f"({'abcdefghi'[k]}) {flab}, {clab}", loc="left", fontsize=8)
            k += 1
        axes[i, 0].set_ylabel("Correlation, r")
        axes[i, 2].set_ylabel("S − L, mean enrichment")
    for ax in axes[-1]:
        ax.set_xlabel("Background design", fontsize=8)
    handles = [
        Line2D(
            [], [], marker=s["marker"], color=s["color"], mec="k", mew=0.5, ls="none", ms=6, label=s["label"]
        )
        for s in PAIR_STYLE.values()
    ]
    handles += [Patch(color="0.82", label="Permutation null, central 95% (line: mean)")]
    fig.legend(handles=handles, loc="lower center", ncol=1, bbox_to_anchor=(0.5, -0.005), fontsize=7.5)
    fig.tight_layout(rect=(0, 0.115 if len(fields) == 2 else 0.075, 1, 1), h_pad=1.0, w_pad=0.6)
    fig.savefig(out_png)
    plt.close(fig)

def _map_panel(ax, z, key_raw, key_w, sig, title, ne_dir, norm, cmap, show_sparse=True, labels=(True, True)):
    lat_e, lon_e = (z["lat_edges"], z["lon_edges"])
    ny, nx = (len(lat_e) - 1, len(lon_e) - 1)
    E = z[key_raw].reshape(ny, nx)
    w = z[key_w].reshape(ny, nx)
    good = np.isfinite(E) & (w >= 0.42)
    sparse_ = (w > 0) & (w < 0.42)
    if show_sparse:
        ax.pcolormesh(lon_e, lat_e, np.where(sparse_, 1.0, np.nan), cmap=ListedColormap(["0.88"]), zorder=1)
    Ep = np.where(good, np.clip(E, norm.vmin * 1.0001, norm.vmax * 0.9999), np.nan)
    pm = ax.pcolormesh(lon_e, lat_e, Ep, cmap=cmap, norm=norm, zorder=2)
    if sig is not None:
        s = sig.reshape(ny, nx) & good
        yy, xx = np.where(s)
        ax.plot(lon_e[xx] + 0.5, lat_e[yy] + 0.5, ".", color="k", ms=1.6, zorder=3)
    M.basemap(ax, ne_dir, labels=False)
    M.region_boxes(ax, color="0.2", lw=0.6)
    M.excl_circle(ax)
    M.receptor(ax, ms=9)
    ax.set_title(title, loc="left", fontsize=8.5)
    if labels[0]:
        ax.set_ylabel("Latitude (°N)")
    if labels[1]:
        ax.set_xlabel("Longitude (°E)")
    return pm

def _cmap():
    return plt.get_cmap("RdBu_r")

def fig8(
    res_dir,
    out_png,
    ne_dir,
    n_days=None,
    field="E",
    sig_field="PSCF",
    layout=(("season", "S"), ("season", "L"), ("wind", "S"), ("wind", "L")),
):
    z, _, _ = _load(res_dir)
    norm = LogNorm(vmin=0.25, vmax=4.0)
    cmap = _cmap()
    fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.9), sharex=True, sharey=True)
    dlab = {"season": "season-matched", "wind": "wind-matched", "unmatched": "unmatched"}
    for k, (ax, (d, c)) in enumerate(zip(axes.ravel(), layout)):
        pm = _map_panel(
            ax,
            z,
            f"{d}__{c}__{field}_raw",
            f"{d}__{c}__w",
            z[f"sig__{d}__{sig_field}__{c}"],
            f"({'abcd'[k]}) {CLS_LAB[c]}, {dlab[d]}",
            ne_dir,
            norm,
            cmap,
            labels=(k % 2 == 0, k >= 2),
        )
    cax = fig.add_axes([0.25, 0.045, 0.5, 0.018])
    cb = fig.colorbar(pm, cax=cax, orientation="horizontal", extend="both")
    cb.set_ticks([0.25, 0.5, 1, 2, 4])
    cb.set_ticklabels(["0.25", "0.5", "1", "2", "4"])
    cb.set_label("Enrichment, observed / expected under random assignment within strata")
    fig.subplots_adjust(left=0.08, right=0.98, top=0.96, bottom=0.14, wspace=0.06, hspace=0.16)
    fig.savefig(out_png)
    plt.close(fig)

def make_all(res_dir, fig_dir, ne_dir, n_days=None):
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig7(res_dir, fig_dir / "Fig7_class_agreement_vs_null.png")
    fig8(res_dir, fig_dir / "Fig8_enrichment_maps.png", ne_dir, n_days=n_days)
    print("trajectory figures written to", fig_dir)

In [ ]:
%%writefile /content/rev_pkg/rev_tables.py
from pathlib import Path
import pandas as pd

REGS = [
    "Syrian Desert",
    "An Nafud",
    "Rub al Khali",
    "Mesopotamia / Fertile Crescent",
    "Ad-Dahna corridor",
    "Northeast Africa",
    "Eastern Arabia / Gulf",
    "Iranian plateau / Sistan",
    "All eight regions",
]
DES = {"unmatched": "Unmatched", "season": "Season-matched", "wind": "Wind-matched"}

def pv(p):
    return "<=0.001" if p < 0.0015 else f"{p:.3f}"

def _reg(res_dir):
    return pd.read_csv(Path(res_dir) / "region_summary_with_nulls.csv")

def _get(r, d, fld, c, reg):
    return r[(r.design == d) & (r.field == fld) & (r["class"] == c) & (r.region == reg)].iloc[0]

def table4(res_dir):
    r = _reg(res_dir)
    rows = []
    for reg in REGS:
        b, bl = (_get(r, "wind", "PSCF", "S", reg), _get(r, "wind", "PSCF", "L", reg))
        nm = [_get(r, d, "PSCF", c, reg).null_mean for d in ("season", "wind") for c in ("S", "L")]
        row = {
            "Region": reg + (" *" if b.excluded_frac > 0.15 and reg != "All eight regions" else ""),
            "Share (%)": f"{100 * b.endpoint_share:.1f}",
            "Cells (W = 1)": f"{int(b.cells_occupied)} ({int(b.cells_W1)})",
            "Days (S / L)": f"{int(b.class_days_crossing)} / {int(bl.class_days_crossing)}",
            "Null mean": f"{sum(nm) / 4:.2f}",
        }
        for d in ("season", "wind"):
            for c in ("S", "L"):
                x = _get(r, d, "PSCF", c, reg)
                row[f"{DES[d]} {c}"] = f"{x['mean']:.2f} ({pv(x.p_region)})"
        rows.append(row)
    return pd.DataFrame(rows)

def table5(res_dir):
    t = pd.read_csv(Path(res_dir) / "table5_counts_by_quintile.csv")
    rows = []
    for _, r in t.iterrows():
        rows.append(
            {
                "Quintile": int(r.quintile),
                "Wind (m/s)": f"{r.wind_lo:.1f}-{r.wind_hi:.1f}",
                "All": f"{int(r.days)} / {int(r.traj)}",
                "S": f"{int(r.S_days)} / {int(r.S_traj)}",
                "L": f"{int(r.L_days)} / {int(r.L_traj)}",
                "Overlap": int(r.both_days),
                "S only": int(r.Sonly_days),
                "L only": int(r.Lonly_days),
                "p_b (S / L)": f"{r.p_S:.3f} / {r.p_L:.3f}",
                "Background (S / L)": f"{int(r.nonS_traj)} / {int(r.nonL_traj)}",
            }
        )
    rows.append(
        {
            "Quintile": "Total",
            "Wind (m/s)": f"{t.wind_lo.min():.1f}-{t.wind_hi.max():.1f}",
            "All": f"{int(t.days.sum())} / {int(t.traj.sum())}",
            "S": f"{int(t.S_days.sum())} / {int(t.S_traj.sum())}",
            "L": f"{int(t.L_days.sum())} / {int(t.L_traj.sum())}",
            "Overlap": int(t.both_days.sum()),
            "S only": int(t.Sonly_days.sum()),
            "L only": int(t.Lonly_days.sum()),
            "p_b (S / L)": f"{t.S_traj.sum() / t.traj.sum():.3f} / {t.L_traj.sum() / t.traj.sum():.3f}",
            "Background (S / L)": f"{int(t.nonS_traj.sum())} / {int(t.nonL_traj.sum())}",
        }
    )
    return pd.DataFrame(rows)

def tableS1(res_dir):
    r = _reg(res_dir)
    out = []
    for panel, fld, cS, cL in [
        ("(a) PSCF", "PSCF", "S", "L"),
        ("(b) PSCF, exclusive classes", "PSCF", "Sonly", "Lonly"),
        ("(c) CWT, daily maximum extinction", "CWT_daymax", "S", "L"),
        ("(d) CWT, dust-code hours", "CWT_hours", "S", "L"),
    ]:
        if r[r.field == fld].empty:
            continue
        for reg in REGS:
            row = {"Panel": panel, "Region": reg}
            for d in DES:
                for lab, c in (("S", cS), ("L", cL)):
                    x = _get(r, d, fld, c, reg)
                    row[f"{DES[d]} {lab}"] = f"{x['mean']:.2f} ({pv(x.p_region)})"
            out.append(row)
    nul = pd.read_csv(Path(res_dir) / "permutation_null_correlations.csv")
    corr = []
    for fld in ["PSCF", "CWT_daymax", "CWT_hours"]:
        for pair, plab in [("L-S", "overlapping"), ("Lonly-Sonly", "exclusive")]:
            row = {"Panel": "(e) Between-class correlation, observed (null mean)", "Region": f"{fld}, {plab}"}
            for d in DES:
                x = nul[(nul.design == d) & (nul.field == fld) & (nul.pair == pair)]
                if x.empty:
                    continue
                x = x.iloc[0]
                row[f"{DES[d]} weighted"] = f"{x.r_obs_weighted:.2f} ({x.null_w_mean:.2f})"
                row[f"{DES[d]} unweighted"] = f"{x.r_obs_unweighted:.2f} ({x.null_u_mean:.2f})"
            corr.append(row)
    return (pd.DataFrame(out), pd.DataFrame(corr))

def make_all(res_dir, out_dir=None):
    out = Path(out_dir or res_dir)
    out.mkdir(parents=True, exist_ok=True)
    t4, t5 = (table4(res_dir), table5(res_dir))
    s1, s1e = tableS1(res_dir)
    t4.to_csv(out / "Table4_regions.csv", index=False)
    t5.to_csv(out / "Table5_counts_by_wind_quintile.csv", index=False)
    s1.to_csv(out / "TableS1_regions_all_designs.csv", index=False)
    s1e.to_csv(out / "TableS1e_correlations.csv", index=False)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    print(t4.to_string(index=False))
    print(t5.to_string(index=False))
    return (t4, t5)

## 3. Results

In [ ]:
# Table 1 (regime statistics) and catalogue checks
import rev_stats as S
d = S.load_class()
print(S.class_counts(d))
print(S.table1(d)[1].round(1).to_string(index=False))

In [ ]:
# Tables 2-3: calibrated expectation and seasonal climatology
%run /content/rev_pkg/rev_calib.py

In [ ]:
# Figures 1-6
import rev_figs
rev_figs.make_all()

In [ ]:
# Tables 4, 5, S1 and Figures 7-8 (trajectory statistics with permutation nulls)
import rev_export, rev_traj_run as RR, rev_traj_figs as TF, rev_tables as TT, paths as P
rev_export.export_traj()
out = P.OUT / "trajectory"
RR.run_all(P.NPZ, P.CLASS_CSV, out, B=1000)
TT.make_all(out)
TF.make_all(out, P.FIGS, P.NE_DIR)